# LeetCode 332 — Reconstruct Itinerary bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def findItinerary(self, tickets: List[List[str]]) -> List[str]:

```

Mục tiêu của bài học:

1. Làm quen với **đồ thị có hướng** (directed graph) từ con số 0: đỉnh là sân bay, cạnh là vé máy bay; hiểu **đa cạnh** (multi-edge — nhiều vé giống hệt nhau) và cách lưu đồ thị bằng **danh sách kề** (adjacency list) với `dict` / `defaultdict`.
2. Hiểu thật chắc **thứ tự từ điển** (lexical / lexicographic order) của chuỗi và của list, vì sao "so cả hành trình như một chuỗi dài" lại tương đương với "so từng sân bay một" trong bài này.
3. Đi từ cách ngây thơ nhất — **thử mọi hoán vị vé** (brute force) — sang **backtracking DFS** (thử, dùng vé, trả vé), và hiểu vì sao lời giải **đầu tiên** tìm được khi thử theo thứ tự từ điển chính là lời giải **nhỏ nhất**.
4. Thấy tận mắt vì sao "cứ đi vé nhỏ nhất" (tham lam) **sai** khi gặp **ngõ cụt** (dead end), và vì sao backtracking có thể bùng nổ theo **giai thừa** với input "bẫy".
5. Học **đường đi Euler** (Euler path) và **thuật toán Hierholzer**: đi tới khi kẹt, ghi sân bay theo **post-order** (ghi khi đã hết vé), rồi **đảo ngược**. Đây là **cách chính**, `O(E log E)`. Hiểu câu then chốt: *"ngõ cụt được ghi vào trước, nên đứng ở cuối hành trình"*.
6. Viết lại Hierholzer bằng **stack** (không đệ quy), tránh các lỗi kinh điển (tham lam không quay lui, ghi sân bay theo pre-order, sort tăng rồi `pop()`, dùng `visited` cho sân bay, quên đảo ngược) và nhận dạng pattern cho các bài anh em như *Valid Arrangement of Pairs*, *Cracking the Safe*.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho một danh sách vé máy bay `tickets`, trong đó `tickets[i] = [from_i, to_i]` là sân bay **khởi hành** và sân bay **đến** của một chuyến bay. Hãy **dựng lại hành trình** (itinerary) theo đúng thứ tự và trả về nó.
>
> Tất cả vé thuộc về một người khởi hành từ `"JFK"`, vì vậy hành trình **phải bắt đầu bằng `"JFK"`**. Nếu có nhiều hành trình hợp lệ, hãy trả về hành trình có **thứ tự từ điển nhỏ nhất** khi đọc như **một chuỗi duy nhất**.
>
> - Ví dụ, hành trình `["JFK", "LGA"]` có thứ tự từ điển nhỏ hơn `["JFK", "LGB"]`.
>
> Bạn có thể giả sử mọi bộ vé đều tạo thành **ít nhất một** hành trình hợp lệ. Bạn **phải dùng tất cả các vé, mỗi vé đúng một lần**.

**Example 1**

```text
Input: tickets = [["MUC","LHR"],["JFK","MUC"],["SFO","SJC"],["LHR","SFO"]]
Output: ["JFK","MUC","LHR","SFO","SJC"]
```

**Example 2**

```text
Input: tickets = [["JFK","SFO"],["JFK","ATL"],["SFO","ATL"],["ATL","JFK"],["ATL","SFO"]]
Output: ["JFK","ATL","JFK","SFO","ATL","SFO"]
Explanation: Một cách dựng khác là ["JFK","SFO","ATL","JFK","ATL","SFO"]
             nhưng nó lớn hơn theo thứ tự từ điển.
```

**Constraints**

- `1 <= tickets.length <= 300`
- `tickets[i].length == 2`
- `from_i.length == 3`
- `to_i.length == 3`
- `from_i` và `to_i` gồm các chữ cái tiếng Anh **in hoa**.
- `from_i != to_i`

### Ký hiệu dùng suốt notebook

- `E` = số vé = `len(tickets)` (tối đa 300). Một hành trình dùng hết vé luôn có **đúng `E + 1` sân bay** (mỗi vé thêm một sân bay đến, cộng sân bay xuất phát `JFK`).
- `V` = số sân bay khác nhau xuất hiện trong vé (`V <= E + 1`).
- Viết `A -> B` nghĩa là "một vé từ `A` bay tới `B`".

### 1.1 Hiểu đề bằng hình

**Example 1** chỉ có một cách đi duy nhất, giống như xếp các mảnh domino nối đuôi nhau:

```text
JFK --> MUC --> LHR --> SFO --> SJC
```

**Example 2** thú vị hơn. Vẽ mỗi vé là một mũi tên:

```text
        JFK ----------> SFO
         ^  \          ^ |
         |   \         | |
         |    v        | v
         +---- ATL ----+ (ATL -> SFO và SFO -> ATL là 2 vé khác chiều)
```

Cụ thể 5 vé: `JFK->SFO`, `JFK->ATL`, `SFO->ATL`, `ATL->JFK`, `ATL->SFO`. Có **hai** cách dùng hết 5 vé xuất phát từ `JFK`:

| Hành trình | Hợp lệ? | Ghi chú |
|---|:---:|---|
| `JFK ATL JFK SFO ATL SFO` | ✅ | đáp án — nhỏ nhất theo từ điển |
| `JFK SFO ATL JFK ATL SFO` | ✅ | hợp lệ nhưng ở vị trí thứ 2, `"SFO" > "ATL"` nên lớn hơn |
| `JFK ATL SFO ATL JFK SFO` | ❓ | trông lạ mắt: hợp lệ hay không? Đếm vé bằng mắt rất dễ nhầm, để code kiểm tra ở cell dưới |
| `JFK ATL JFK` | ❌ | chưa dùng hết vé |
| `SFO ATL JFK ATL SFO JFK` | ❌ | không bắt đầu từ `JFK` (và có vé `SFO->JFK` không tồn tại) |

Hàng thứ 3 cố tình để bạn nghi ngờ — cell code dưới sẽ trả lời giúp ta. Bài học ở đây: **đừng kiểm tra bằng mắt**, hãy viết một hàm kiểm tra.

Vài suy luận nhỏ mà quan trọng:

- Một sân bay **được ghé lại nhiều lần** (`JFK` xuất hiện 2 lần trong đáp án Example 2). Thứ "chỉ được dùng một lần" là **vé**, không phải sân bay.
- Có thể có **vé trùng nhau** (ví dụ hai vé `JFK->ATL`): khi đó ta phải dùng **cả hai**, mỗi vé một lần.
- Mọi hành trình hợp lệ đều dài đúng `E + 1`, nên khi so sánh hai hành trình ta không phải lo "cái ngắn hơn thắng" — chỉ có so từng vị trí.

Cell dưới định nghĩa các ví dụ của đề, thêm một ví dụ có **ngõ cụt** mà ta sẽ dùng rất nhiều (`ex3`), và một hàm `is_valid_itinerary` kiểm tra một hành trình có hợp lệ không. Hàm dùng `Counter` (bộ đếm) — một loại `dict` đặc biệt để đếm số lần xuất hiện; mục 2.1 sẽ giảng kỹ.

In [1]:
from typing import List
from collections import defaultdict, Counter
from itertools import permutations
import heapq
import math
import random
import time

ex1 = [["MUC", "LHR"], ["JFK", "MUC"], ["SFO", "SJC"], ["LHR", "SFO"]]
ex2 = [["JFK", "SFO"], ["JFK", "ATL"], ["SFO", "ATL"], ["ATL", "JFK"], ["ATL", "SFO"]]
ex3 = [["JFK", "KUL"], ["JFK", "NRT"], ["NRT", "JFK"]]          # có ngõ cụt KUL
ans1 = ["JFK", "MUC", "LHR", "SFO", "SJC"]
ans2 = ["JFK", "ATL", "JFK", "SFO", "ATL", "SFO"]
ans3 = ["JFK", "NRT", "JFK", "KUL"]


def is_valid_itinerary(tickets, route):
    # Hợp lệ = bắt đầu ở JFK, dài E+1, và mỗi bước A->B tiêu đúng một vé A->B còn lại.
    if not route or route[0] != "JFK" or len(route) != len(tickets) + 1:
        return False
    remaining = Counter(tuple(t) for t in tickets)   # vé -> số tờ còn lại
    for a, b in zip(route, route[1:]):
        if remaining[(a, b)] == 0:
            return False
        remaining[(a, b)] -= 1
    return True


print("đáp án Ex2          :", is_valid_itinerary(ex2, ans2))
print("cách dựng khác Ex2  :", is_valid_itinerary(ex2, ["JFK", "SFO", "ATL", "JFK", "ATL", "SFO"]))
print("hàng thứ 3 của bảng :", is_valid_itinerary(ex2, ["JFK", "ATL", "SFO", "ATL", "JFK", "SFO"]))
print("chưa dùng hết vé    :", is_valid_itinerary(ex2, ["JFK", "ATL", "JFK"]))
print("không bắt đầu JFK   :", is_valid_itinerary(ex2, ["SFO", "ATL", "JFK", "ATL", "SFO", "JFK"]))
print("đáp án Ex1, Ex3     :", is_valid_itinerary(ex1, ans1), is_valid_itinerary(ex3, ans3))

đáp án Ex2          : True
cách dựng khác Ex2  : True
hàng thứ 3 của bảng : True
chưa dùng hết vé    : False
không bắt đầu JFK   : False
đáp án Ex1, Ex3     : True True


**Bất ngờ chưa?** Hàng thứ 3 `JFK ATL SFO ATL JFK SFO` cũng **hợp lệ**! Vậy Example 2 thực ra có **ba** hành trình hợp lệ, không phải hai như đề nhắc tới:

```text
JFK ATL JFK SFO ATL SFO   <- nhỏ nhất (đáp án)
JFK ATL SFO ATL JFK SFO
JFK SFO ATL JFK ATL SFO
```

So hai hàng đầu: giống nhau ở `JFK ATL`, khác nhau ở vị trí thứ 3: `"JFK" < "SFO"` nên hàng đầu nhỏ hơn. Đề chỉ nêu một "cách dựng khác" làm ví dụ, không khẳng định đó là tất cả. Đây là lý do notebook này luôn **kiểm chứng bằng code** thay vì tin vào cảm giác.

Còn `ex3` (vé `JFK->KUL`, `JFK->NRT`, `NRT->JFK`) là ví dụ **ngõ cụt**: `KUL` không có vé nào bay đi. Nếu từ `JFK` ta vội bay tới `KUL` (vì `"KUL" < "NRT"`), ta kẹt luôn ở `KUL` trong khi còn 2 vé chưa dùng. Đáp án đúng phải là `JFK NRT JFK KUL` — "đi vòng" qua `NRT` trước, để ngõ cụt `KUL` lại **sau cùng**. Ví dụ nhỏ xíu này là chìa khóa của cả bài.

## 2. Nền tảng

### 2.1 `dict`, `defaultdict` và `Counter`

**`dict`** (từ điển) lưu các cặp `key -> value`. Ta sẽ dùng key là **tên sân bay**, value là **danh sách các nơi có thể bay tới**.

Vấn đề nhỏ: với `dict` thường, lần đầu gặp một sân bay ta phải tự tạo list rỗng, nếu không sẽ bị lỗi `KeyError`. **`defaultdict(list)`** làm việc đó tự động: hễ truy cập một key chưa có, nó tự tạo `[]` cho key đó.

**`Counter`** là một `defaultdict` chuyên **đếm**: truy cập key chưa có thì trả `0`. Hàm `is_valid_itinerary` ở trên dùng nó để đếm "mỗi loại vé còn bao nhiêu tờ".

Thử từng cái trong cell dưới.

In [2]:
d = {}
d["JFK"] = ["SFO"]
print("dict thường      :", d)
try:
    d["ATL"].append("JFK")              # ATL chưa có -> lỗi
except KeyError as e:
    print("KeyError với key :", e)

g = defaultdict(list)
g["JFK"].append("SFO")                  # tự tạo g["JFK"] = [] rồi append
g["JFK"].append("ATL")
g["ATL"].append("JFK")
print("defaultdict      :", dict(g))
print("truy cập 'XYZ'   :", g["XYZ"], "-> giờ g có key XYZ:", "XYZ" in g)

c = Counter([("JFK", "ATL"), ("JFK", "ATL"), ("ATL", "JFK")])
print("Counter          :", c)
print("vé chưa có       :", c[("SFO", "LHR")])

dict thường      : {'JFK': ['SFO']}
KeyError với key : 'ATL'
defaultdict      : {'JFK': ['SFO', 'ATL'], 'ATL': ['JFK']}
truy cập 'XYZ'   : [] -> giờ g có key XYZ: True
Counter          : Counter({('JFK', 'ATL'): 2, ('ATL', 'JFK'): 1})
vé chưa có       : 0


Lưu ý dòng `truy cập 'XYZ'`: chỉ cần **đọc** `g["XYZ"]` là `defaultdict` đã **tạo luôn** key `XYZ`. Đây là tính năng chứ không phải lỗi, nhưng nên nhớ: nếu muốn "chỉ hỏi, không tạo", dùng `g.get("XYZ", [])` hoặc `"XYZ" in g`.

### 2.2 Đồ thị có hướng và danh sách kề

Một **đồ thị** (graph) gồm các **đỉnh** (vertex — ở đây là sân bay) và các **cạnh** (edge — ở đây là vé). Vé có chiều (`JFK->ATL` khác `ATL->JFK`), nên đây là **đồ thị có hướng**.

**Danh sách kề** (adjacency list): với mỗi đỉnh `u`, lưu list các đỉnh `v` mà có cạnh `u -> v`. Với Example 2:

```text
JFK : [SFO, ATL]
SFO : [ATL]
ATL : [JFK, SFO]
```

Đồ thị của bài này có thể có **đa cạnh** (multi-edge): hai vé `JFK->ATL` giống hệt nhau thì `ATL` xuất hiện **hai lần** trong list của `JFK`. Vì vậy ta dùng `list` (giữ trùng lặp), **không** dùng `set` (sẽ nuốt mất một vé).

Thuật ngữ đi kèm:

- **Bậc ra** (out-degree) của `u` = số vé bay **đi** từ `u` = `len(adj[u])`.
- **Bậc vào** (in-degree) của `v` = số vé bay **tới** `v`.

In [3]:
def build_adj(tickets):
    adj = defaultdict(list)
    for a, b in tickets:
        adj[a].append(b)
    return adj


adj2 = build_adj(ex2)
print("Danh sách kề Example 2:")
for airport in adj2:
    print(f"  {airport} : {adj2[airport]}")

multi = [["JFK", "ATL"], ["ATL", "JFK"], ["JFK", "ATL"], ["ATL", "JFK"]]
print("Đa cạnh (giữ list)    :", dict(build_adj(multi)))
print("Nếu lỡ dùng set       :", {a: {b for x, b in multi if x == a} for a, _ in multi}, "<- mất vé!")
print("Bậc ra của JFK (Ex2)  :", len(adj2["JFK"]))
print("Bậc vào của SFO (Ex2) :", sum(1 for a, b in ex2 if b == "SFO"))

Danh sách kề Example 2:
  JFK : ['SFO', 'ATL']
  SFO : ['ATL']
  ATL : ['JFK', 'SFO']
Đa cạnh (giữ list)    : {'JFK': ['ATL', 'ATL'], 'ATL': ['JFK', 'JFK']}
Nếu lỡ dùng set       : {'JFK': {'ATL'}, 'ATL': {'JFK'}} <- mất vé!
Bậc ra của JFK (Ex2)  : 2
Bậc vào của SFO (Ex2) : 2


### 2.3 Thứ tự từ điển: so sánh chuỗi và so sánh list

Python so sánh hai chuỗi giống cách tra từ điển: so **ký tự đầu tiên khác nhau** (theo mã Unicode, với chữ in hoa thì đơn giản là `A < B < ... < Z`).

- `"ATL" < "JFK"` vì `'A' < 'J'`.
- `"LGA" < "LGB"` vì hai ký tự đầu giống nhau, ký tự thứ 3 `'A' < 'B'`.

**List cũng được so sánh theo thứ tự từ điển**: so phần tử đầu tiên khác nhau. Ví dụ `["JFK", "ATL", "JFK"] < ["JFK", "SFO", "ATL"]` vì phần tử đầu giống nhau, phần tử thứ 2 `"ATL" < "SFO"`.

Đề nói "đọc như **một chuỗi duy nhất**" — tức nối hết lại: `"JFKATLJFK..."`. Trong bài này, **mọi mã sân bay dài đúng 3 ký tự** và **mọi hành trình dài đúng `E + 1`**, nên chuỗi nối lại có từng "khúc 3 ký tự" thẳng hàng nhau. So sánh chuỗi dài vì thế **cho cùng kết quả** với so sánh list từng phần tử. Nếu độ dài mã khác nhau thì điều này có thể sai — xem dòng cuối cell dưới.

In [4]:
print('"ATL" < "JFK"         :', "ATL" < "JFK")
print('"LGA" < "LGB"         :', "LGA" < "LGB")
print("sorted mã sân bay     :", sorted(["SFO", "JFK", "ATL", "KUL", "NRT"]))

r1 = ["JFK", "ATL", "JFK", "SFO", "ATL", "SFO"]
r2 = ["JFK", "SFO", "ATL", "JFK", "ATL", "SFO"]
print("so list    r1 < r2    :", r1 < r2)
print("so chuỗi   r1 < r2    :", "".join(r1) < "".join(r2), " ->", "".join(r1), "vs", "".join(r2))

# Khi mã KHÔNG cùng độ dài, hai cách so có thể khác nhau:
a, b = ["A", "BC"], ["AB", "C"]
print("mã lệch độ dài        : list a<b =", a < b, "| chuỗi bằng nhau =", "".join(a) == "".join(b))

"ATL" < "JFK"         : True
"LGA" < "LGB"         : True
sorted mã sân bay     : ['ATL', 'JFK', 'KUL', 'NRT', 'SFO']
so list    r1 < r2    : True
so chuỗi   r1 < r2    : True  -> JFKATLJFKSFOATLSFO vs JFKSFOATLJFKATLSFO
mã lệch độ dài        : list a<b = True | chuỗi bằng nhau = True


### 2.4 Sắp xếp danh sách kề, và vì sao lấy từ **cuối** list

Ta muốn "từ mỗi sân bay, thử nơi đến **nhỏ nhất** trước". Cách tự nhiên: sắp xếp mỗi list kề tăng dần rồi lấy phần tử đầu `pop(0)`. Nhưng `pop(0)` phải **dời cả list** sang trái một ô → tốn `O(k)` với list dài `k`. Ngược lại `pop()` (lấy phần tử **cuối**) chỉ tốn `O(1)`.

Mẹo kinh điển: **sắp xếp giảm dần** (`reverse=True`), khi đó phần tử nhỏ nhất nằm ở **cuối**, và `pop()` lấy ra đúng nó với chi phí `O(1)`.

```text
tăng dần  : [ATL, JFK, SFO]   pop(0) -> ATL  (chậm: dời 2 phần tử còn lại)
giảm dần  : [SFO, JFK, ATL]   pop()  -> ATL  (nhanh: chỉ cắt đuôi)
```

Một cách gọn để sắp xếp tất cả list kề một lần: **sort cả danh sách vé** giảm dần rồi mới `append`. Vì vé được sort theo `(from, to)`, các nơi đến của cùng một sân bay sẽ được append theo thứ tự giảm dần.

In [5]:
dests = ["JFK", "SFO", "ATL"]
inc = sorted(dests)
dec = sorted(dests, reverse=True)
print("tăng dần:", inc, end=" ")
print("-> pop(0) =", inc.pop(0), "| còn", inc)
print("giảm dần:", dec, end=" ")
print("-> pop()  =", dec.pop(), "| còn", dec)

adj_desc = defaultdict(list)
for a, b in sorted(ex2, reverse=True):          # sort cả vé giảm dần theo (from, to)
    adj_desc[a].append(b)
print("adj giảm dần Ex2:", dict(adj_desc))

big = list(range(100_000))
t0 = time.perf_counter()
for _ in range(20_000):
    big.pop(0)
t1 = time.perf_counter()
big = list(range(100_000))
for _ in range(20_000):
    big.pop()
t2 = time.perf_counter()
print(f"20 000 lần pop(0): {t1 - t0:.4f}s | 20 000 lần pop(): {t2 - t1:.4f}s")

tăng dần: ['ATL', 'JFK', 'SFO'] -> pop(0) = ATL | còn ['JFK', 'SFO']
giảm dần: ['SFO', 'JFK', 'ATL'] -> pop()  = ATL | còn ['SFO', 'JFK']
adj giảm dần Ex2: {'SFO': ['ATL'], 'JFK': ['SFO', 'ATL'], 'ATL': ['SFO', 'JFK']}


20 000 lần pop(0): 0.3173s | 20 000 lần pop(): 0.0035s


### 2.5 Đệ quy và DFS: đi sâu theo thứ tự từ điển

**Đệ quy** (recursion) là hàm tự gọi lại chính nó trên một bài toán con. **DFS** (Depth-First Search — duyệt theo chiều sâu) là cách duyệt "đi sâu hết mức có thể theo một nhánh, rồi mới quay lại thử nhánh khác" — và đệ quy là cách viết DFS tự nhiên nhất.

Ví dụ mini (chưa phải bài chính): một "cây đường bay" nhỏ, liệt kê **mọi đường đi** từ `JFK` tới khi hết đường. Nếu ở mỗi sân bay ta thử các nơi đến theo **thứ tự từ điển**, thì các đường đi hoàn chỉnh được in ra **theo đúng thứ tự từ điển**:

```text
          JFK
         /    \
       ATL     SFO
      /   \      \
    BOS   LAX    SEA
```

Lý do: mọi đường đi qua `ATL` (nhỏ hơn) đều được liệt kê **xong hết** trước khi DFS chuyển sang nhánh `SFO`. Hai đường có chung phần đầu thì "chỗ rẽ" đầu tiên quyết định thứ tự — đúng định nghĩa thứ tự từ điển. **Quan sát này là nền tảng cho Cách 2.**

In [6]:
tree = {"JFK": ["SFO", "ATL"], "ATL": ["LAX", "BOS"], "SFO": ["SEA"]}


def list_paths(u, path, out):
    path.append(u)
    children = sorted(tree.get(u, []))          # thử theo thứ tự từ điển
    if not children:
        out.append(list(path))                  # hết đường: ghi lại một đường hoàn chỉnh
    for v in children:
        list_paths(v, path, out)                # đi sâu vào nhánh v
    path.pop()                                  # quay lui: bỏ u khỏi đường đang xét


paths = []
list_paths("JFK", [], paths)
for p in paths:
    print(" -> ".join(p))
print("Thứ tự in ra đã được sắp xếp sẵn?", paths == sorted(paths))

JFK -> ATL -> BOS
JFK -> ATL -> LAX
JFK -> SFO -> SEA
Thứ tự in ra đã được sắp xếp sẵn? True


### 2.6 Backtracking: "dùng rồi trả"

**Backtracking** (quay lui) là DFS trên **các lựa chọn**: ở mỗi bước, chọn một phương án còn được phép, **đánh dấu đã dùng**, đi tiếp; nếu nhánh đó không đi tới đâu thì **trả lại** (bỏ đánh dấu) và thử phương án khác.

Ví dụ mini kinh điển: liệt kê mọi hoán vị (cách xếp thứ tự) của `["A", "B", "C"]`. Mảng `used[i]` cho biết phần tử `i` đã nằm trong hoán vị đang xây chưa. Ba dòng quan trọng nhất luôn đi thành bộ:

```python
used[i] = True;  cur.append(x)     # dùng
dfs()                               # đi tiếp
cur.pop();       used[i] = False    # trả
```

Bài 332 sẽ dùng đúng khuôn này, chỉ thay "phần tử" bằng "vé".

In [7]:
def permutations_backtrack(items):
    used = [False] * len(items)
    cur, out = [], []

    def dfs():
        if len(cur) == len(items):
            out.append("".join(cur))
            return
        for i, x in enumerate(items):
            if used[i]:
                continue
            used[i] = True; cur.append(x)      # dùng
            dfs()                              # đi tiếp
            cur.pop(); used[i] = False         # trả

    dfs()
    return out


print(permutations_backtrack(["A", "B", "C"]))

['ABC', 'ACB', 'BAC', 'BCA', 'CAB', 'CBA']


## 3. Cách 1 — Brute force: thử mọi thứ tự vé

Cách người mới nghĩ ra đầu tiên: *"Hành trình chẳng qua là xếp các vé theo một thứ tự nào đó. Vậy thử **mọi thứ tự**, giữ những thứ tự 'nối đuôi' được, rồi chọn cái nhỏ nhất."*

Một thứ tự vé `t0, t1, ..., t(E-1)` cho ra hành trình hợp lệ khi:

1. Vé đầu xuất phát từ `JFK`: `t0[0] == "JFK"`.
2. Vé sau nối với vé trước: `t(i)[1] == t(i+1)[0]` với mọi `i`.

Khi đó hành trình là `[t0[0], t0[1], t1[1], ..., t(E-1)[1]]`.

Để sinh mọi thứ tự, Python có sẵn `itertools.permutations` (chính là thứ ta vừa tự viết bằng backtracking ở 2.6).

In [8]:
print(list(permutations(["A", "B", "C"])))
print("Số hoán vị của 3 vé:", len(list(permutations(range(3)))), "= 3! =", math.factorial(3))


def itinerary_bruteforce(tickets):
    best = None
    for order in permutations(tickets):
        if order[0][0] != "JFK":
            continue
        if all(order[i][1] == order[i + 1][0] for i in range(len(order) - 1)):
            route = [order[0][0]] + [b for a, b in order]
            if best is None or route < best:
                best = route
    return best


for name, t, ans in [("Ex1", ex1, ans1), ("Ex2", ex2, ans2), ("Ex3", ex3, ans3)]:
    got = itinerary_bruteforce(t)
    print(name, got, "OK" if got == ans else "SAI")

[('A', 'B', 'C'), ('A', 'C', 'B'), ('B', 'A', 'C'), ('B', 'C', 'A'), ('C', 'A', 'B'), ('C', 'B', 'A')]
Số hoán vị của 3 vé: 6 = 3! = 6
Ex1 ['JFK', 'MUC', 'LHR', 'SFO', 'SJC'] OK
Ex2 ['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO'] OK
Ex3 ['JFK', 'NRT', 'JFK', 'KUL'] OK


### 3.1 Brute force tốn bao nhiêu?

Với `E` vé có `E!` (giai thừa) thứ tự. Mỗi thứ tự tốn `O(E)` để kiểm tra, nên tổng cộng `O(E! · E)`.

Cell dưới đếm, với Example 2, có bao nhiêu thứ tự được thử và bao nhiêu thứ tự hợp lệ — rồi in bảng `E!` cho thấy con số "nổ" nhanh thế nào. Với `E = 300` (giới hạn của đề), `300!` có hơn 600 chữ số: không máy tính nào chạy nổi.

In [9]:
def count_orders(tickets):
    total = valid = 0
    for order in permutations(tickets):
        total += 1
        if order[0][0] == "JFK" and all(order[i][1] == order[i + 1][0] for i in range(len(order) - 1)):
            valid += 1
    return total, valid


print("Ex2: (số thứ tự đã thử, số thứ tự hợp lệ) =", count_orders(ex2))
for E in [5, 8, 10, 15, 20]:
    print(f"E = {E:>2}: E! = {math.factorial(E):,}")
print("E = 300: 300! có", len(str(math.factorial(300))), "chữ số")

Ex2: (số thứ tự đã thử, số thứ tự hợp lệ) = (120, 3)
E =  5: E! = 120
E =  8: E! = 40,320
E = 10: E! = 3,628,800
E = 15: E! = 1,307,674,368,000
E = 20: E! = 2,432,902,008,176,640,000
E = 300: 300! có 615 chữ số


### 3.2 Brute force vướng ở đâu? → Mấu chốt cải tiến

Nhìn kỹ: brute force sinh **cả** thứ tự rồi mới kiểm tra. Nhưng phần lớn thứ tự đã "chết" ngay từ vé đầu tiên (không xuất phát từ `JFK`) hoặc vé thứ hai (không nối được). Hai lãng phí lớn:

1. **Không cắt nhánh sớm**: nếu vé thứ hai đã không nối với vé thứ nhất, mọi `(E-2)!` cách xếp phần còn lại đều vô ích.
2. **Không tận dụng thứ tự từ điển**: brute force duyệt hết rồi mới chọn `min`.

Mấu chốt cải tiến:

- **Xây hành trình từng bước**: đang đứng ở sân bay `u`, chỉ thử những vé **bắt đầu từ `u`** (lấy từ danh sách kề) → không bao giờ sinh thứ tự "không nối".
- **Thử nơi đến theo thứ tự từ điển** → như ta thấy ở 2.5, các hành trình hoàn chỉnh sẽ hiện ra theo thứ tự tăng dần, nên **cái đầu tiên tìm được là cái nhỏ nhất**, dừng luôn.

Đó chính là Cách 2.

## 4. Cách 2 — Backtracking DFS theo thứ tự từ điển

### 4.1 Trước hết: vì sao "cứ đi vé nhỏ nhất" (tham lam) là SAI?

Ý tưởng hấp dẫn nhất: *"Đứng ở đâu thì đi vé tới nơi nhỏ nhất, khỏi quay lui."* Đây gọi là **tham lam** (greedy). Với Example 1 và 2 nó đúng, nhưng với `ex3` nó hỏng ngay:

```text
Đứng ở JFK: vé còn lại tới [KUL, NRT] -> chọn KUL (nhỏ nhất)
Đứng ở KUL: không còn vé nào bay đi  -> KẸT, còn 2 vé chưa dùng!
```

Nhỏ nhất **tại chỗ** không có nghĩa là đi được **tới đích**. Ta cần một cơ chế "thấy kẹt thì lùi lại, chọn khác" — tức backtracking.

In [10]:
def itinerary_greedy_wrong(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets, reverse=True):
        adj[a].append(b)
    route = ["JFK"]
    while adj[route[-1]]:
        route.append(adj[route[-1]].pop())     # luôn lấy nơi đến nhỏ nhất còn lại
    return route


for name, t in [("Ex1", ex1), ("Ex2", ex2), ("Ex3", ex3)]:
    r = itinerary_greedy_wrong(t)
    print(f"{name}: {r} | dùng {len(r) - 1}/{len(t)} vé | hợp lệ: {is_valid_itinerary(t, r)}")

Ex1: ['JFK', 'MUC', 'LHR', 'SFO', 'SJC'] | dùng 4/4 vé | hợp lệ: True
Ex2: ['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO'] | dùng 5/5 vé | hợp lệ: True
Ex3: ['JFK', 'KUL'] | dùng 1/3 vé | hợp lệ: False


### 4.2 Thuật toán backtracking

Chuẩn bị:

- `adj[u]` = list nơi đến từ `u`, **sắp xếp tăng dần** (đa cạnh thì lặp lại).
- `used[u][i]` = vé thứ `i` trong `adj[u]` đã dùng chưa.
- `route` = hành trình đang xây, bắt đầu `["JFK"]`.

Hàm `dfs(u)` trả về `True` nếu **từ trạng thái hiện tại** có thể dùng hết vé:

1. Nếu `len(route) == E + 1` → đã dùng đủ `E` vé → `True`.
2. Duyệt các vé `u -> v` **theo thứ tự tăng dần**, bỏ qua vé đã dùng:
   - **Dùng**: `used[u][i] = True`, `route.append(v)`.
   - **Đi tiếp**: nếu `dfs(v)` trả `True` → trả `True` luôn (không trả vé, giữ nguyên `route`).
   - **Trả**: `route.pop()`, `used[u][i] = False`.
3. Thử hết mà không được → `False`.

**Vì sao lời giải đầu tiên là nhỏ nhất?** Mọi hành trình hoàn chỉnh dài đúng `E + 1`. Lấy hai hành trình hoàn chỉnh `P < Q`: chúng giống nhau tới vị trí `k - 1` rồi `P[k] < Q[k]`. Khi DFS đứng ở cuối phần chung đó, nó thử nơi đến `P[k]` **trước** `Q[k]`, và chỉ chuyển sang `Q[k]` khi đã duyệt **xong toàn bộ** nhánh `P[k]` — tức là đã gặp `P` (hoặc một hành trình còn nhỏ hơn) trước. Vậy DFS không thể gặp `Q` trước `P`: hành trình đầu tiên gặp được nhỏ hơn mọi hành trình khác.

In [11]:
def itinerary_backtracking(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets):                 # tăng dần
        adj[a].append(b)
    used = {a: [False] * len(adj[a]) for a in list(adj)}
    E = len(tickets)
    route = ["JFK"]

    def dfs(u):
        if len(route) == E + 1:
            return True
        for i, v in enumerate(adj.get(u, [])):
            if used[u][i]:
                continue
            used[u][i] = True; route.append(v)          # dùng vé u->v
            if dfs(v):
                return True                             # tìm thấy: dừng, giữ nguyên route
            route.pop(); used[u][i] = False             # trả vé
        return False

    dfs("JFK")
    return route


for name, t, ans in [("Ex1", ex1, ans1), ("Ex2", ex2, ans2), ("Ex3", ex3, ans3)]:
    got = itinerary_backtracking(t)
    print(name, got, "OK" if got == ans else "SAI")

Ex1 ['JFK', 'MUC', 'LHR', 'SFO', 'SJC'] OK
Ex2 ['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO'] OK
Ex3 ['JFK', 'NRT', 'JFK', 'KUL'] OK


### 4.3 Trace backtracking trên `ex3` (có ngõ cụt)

`adj` tăng dần: `JFK: [KUL, NRT]`, `NRT: [JFK]`. `E = 3`, cần `route` dài 4.

| Bước | Đứng ở | Hành động | `route` sau bước | Ghi chú |
|---:|---|---|---|---|
| 1 | JFK | dùng vé `JFK->KUL` | `JFK KUL` | thử nơi nhỏ nhất trước |
| 2 | KUL | không có vé → `False` | `JFK KUL` | ngõ cụt, mới dùng 1/3 vé |
| 3 | JFK | **trả** vé `JFK->KUL` | `JFK` | quay lui |
| 4 | JFK | dùng vé `JFK->NRT` | `JFK NRT` | thử nơi tiếp theo |
| 5 | NRT | dùng vé `NRT->JFK` | `JFK NRT JFK` | |
| 6 | JFK | dùng vé `JFK->KUL` | `JFK NRT JFK KUL` | vé này đã được trả ở bước 3 nên dùng lại được |
| 7 | KUL | `len(route) == 4` → `True` | `JFK NRT JFK KUL` | dừng, đây là đáp án |

Hàm `trace_backtracking` in đúng các bước này (thụt lề theo độ sâu đệ quy). Sau đó chạy thêm Example 2 để thấy: ở Example 2 **không cần lùi lần nào** — tham lam tình cờ đúng.

In [12]:
def trace_backtracking(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets):
        adj[a].append(b)
    used = {a: [False] * len(adj[a]) for a in list(adj)}
    E = len(tickets)
    route = ["JFK"]

    def dfs(u, depth):
        pad = "  " * depth
        if len(route) == E + 1:
            print(f"{pad}đủ {E} vé -> True, route = {' '.join(route)}")
            return True
        if not any(not x for x in used.get(u, [])):
            print(f"{pad}ở {u}: hết vé, mới dùng {len(route) - 1}/{E} -> False")
        for i, v in enumerate(adj.get(u, [])):
            if used[u][i]:
                continue
            used[u][i] = True; route.append(v)
            print(f"{pad}ở {u}: dùng vé {u}->{v}   route = {' '.join(route)}")
            if dfs(v, depth + 1):
                return True
            route.pop(); used[u][i] = False
            print(f"{pad}ở {u}: TRẢ vé {u}->{v}    route = {' '.join(route)}")
        return False

    dfs("JFK", 0)
    return route


print("=== ex3 ===")
trace_backtracking(ex3)
print("=== Example 2 ===")
trace_backtracking(ex2)

=== ex3 ===
ở JFK: dùng vé JFK->KUL   route = JFK KUL
  ở KUL: hết vé, mới dùng 1/3 -> False
ở JFK: TRẢ vé JFK->KUL    route = JFK
ở JFK: dùng vé JFK->NRT   route = JFK NRT
  ở NRT: dùng vé NRT->JFK   route = JFK NRT JFK
    ở JFK: dùng vé JFK->KUL   route = JFK NRT JFK KUL
      đủ 3 vé -> True, route = JFK NRT JFK KUL
=== Example 2 ===
ở JFK: dùng vé JFK->ATL   route = JFK ATL
  ở ATL: dùng vé ATL->JFK   route = JFK ATL JFK
    ở JFK: dùng vé JFK->SFO   route = JFK ATL JFK SFO
      ở SFO: dùng vé SFO->ATL   route = JFK ATL JFK SFO ATL
        ở ATL: dùng vé ATL->SFO   route = JFK ATL JFK SFO ATL SFO
          đủ 5 vé -> True, route = JFK ATL JFK SFO ATL SFO


['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO']

### 4.4 Vé trùng nhau và mẹo "bỏ qua nơi đến vừa thất bại"

Khi có hai vé **giống hệt** `u -> v`, chúng hoàn toàn **thay thế được cho nhau**. Nếu đã dùng tờ thứ nhất mà thất bại, thử tờ thứ hai chắc chắn cũng thất bại y hệt — phí công. Mẹo: nhớ nơi đến vừa thất bại `prev`; gặp lại `v == prev` (list đã sắp xếp nên các vé trùng nằm cạnh nhau) thì bỏ qua.

Cell dưới đếm số lần gọi `dfs` của hai phiên bản trên bộ vé `dup_trap(k)`:

- `JFK -> AAA`, `JFK -> CCC`, `CCC -> JFK` (vòng nhỏ qua `CCC`), `AAA -> ZZZ` (ngõ cụt);
- cộng thêm `k` **bản sao giống hệt** của vòng `AAA -> PPP -> AAA`.

Không có vé nào từ vùng `AAA` quay về `JFK`, nên đáp án bắt buộc đi vòng `CCC` trước. Backtracking lại thử `AAA` trước (vì `"AAA" < "CCC"`), lạc vào vùng `AAA` và chỉ quay ra khi đã thử hết mọi cách đi trong đó. Không bỏ trùng thì nó coi `k` tờ `AAA->PPP` (và `k` tờ `PPP->AAA`) là `k` lựa chọn **khác nhau**, nên số cách thử lên tới cỡ `(k!)²`. Có bỏ trùng thì mỗi sân bay chỉ còn tối đa 2 lựa chọn thật sự (`PPP` hoặc `ZZZ`).

In [13]:
def count_backtracking_calls(tickets, skip_duplicates):
    adj = defaultdict(list)
    for a, b in sorted(tickets):
        adj[a].append(b)
    used = {a: [False] * len(adj[a]) for a in list(adj)}
    E = len(tickets)
    route = ["JFK"]
    calls = 0

    def dfs(u):
        nonlocal calls
        calls += 1
        if len(route) == E + 1:
            return True
        prev = None
        for i, v in enumerate(adj.get(u, [])):
            if used[u][i] or (skip_duplicates and v == prev):
                continue
            used[u][i] = True; route.append(v)
            if dfs(v):
                return True
            route.pop(); used[u][i] = False
            prev = v                                    # v vừa thất bại
        return False

    dfs("JFK")
    return route, calls


def dup_trap(k):
    return [["JFK", "AAA"], ["JFK", "CCC"], ["CCC", "JFK"], ["AAA", "ZZZ"]] + [["AAA", "PPP"], ["PPP", "AAA"]] * k


print(" k | không bỏ trùng | có bỏ trùng")
for k in range(1, 6):
    r1, c1 = count_backtracking_calls(dup_trap(k), False)
    r2, c2 = count_backtracking_calls(dup_trap(k), True)
    assert r1 == r2 and is_valid_itinerary(dup_trap(k), r1)
    print(f"{k:>2} | {c1:>14,} | {c2:>11,}")
print("hành trình với k=2:", " ".join(count_backtracking_calls(dup_trap(2), True)[0]))

dup = [["JFK", "AAA"]] + [["JFK", "BBB"]] * 3 + [["BBB", "JFK"]] * 3    # dùng lại ở mục 9

 k | không bỏ trùng | có bỏ trùng
 1 |             12 |          12
 2 |             33 |          17
 3 |            232 |          22
 4 |          3,555 |          27
 5 |         88,572 |          32
hành trình với k=2: JFK CCC JFK AAA PPP AAA PPP AAA ZZZ


### 4.5 Trường hợp xấu: backtracking bùng nổ theo giai thừa

Bảng trên cho thấy mẹo bỏ trùng cứu được trường hợp **các vòng giống hệt nhau** (88 572 lần gọi xuống còn 32 khi `k = 5`). Nhưng chỉ cần các vòng **khác tên nhau** một chút là mẹo đó vô dụng. Hãy dựng một input "bẫy" như vậy:

- `JFK -> AAA` và `JFK -> CCC`, `CCC -> JFK` (một vòng nhỏ qua `CCC`).
- Tại `AAA` có `k` vòng nhỏ khác nhau `AAA -> P_i -> AAA`, và một vé `AAA -> ZZZ` (ngõ cụt).

Không có vé nào từ vùng `AAA` quay về `JFK`, nên hành trình đúng **bắt buộc** đi vòng `CCC` trước: `JFK CCC JFK AAA ... ZZZ`. Nhưng backtracking thử `AAA` trước (vì `"AAA" < "CCC"`), rồi đi lang thang qua **mọi thứ tự** của `k` vòng `P_i` — khoảng `k!` thứ tự — và cái nào cuối cùng cũng thất bại vì còn sót vòng `CCC`. Chỉ khi đó nó mới chịu lùi về `JFK` để thử `CCC`.

In [14]:
def trap_tickets(k):
    t = [["JFK", "AAA"], ["JFK", "CCC"], ["CCC", "JFK"], ["AAA", "ZZZ"]]
    for i in range(k):
        p = "P" + chr(ord("A") + i) + "A"                 # PAA, PBA, PCA, ...
        t += [["AAA", p], [p, "AAA"]]
    return t


print(" k |   E | số lần gọi dfs | k!")
for k in range(3, 9):
    t = trap_tickets(k)
    route, calls = count_backtracking_calls(t, True)
    assert is_valid_itinerary(t, route)
    print(f"{k:>2} | {len(t):>3} | {calls:>14,} | {math.factorial(k):,}")
print("hành trình với k=3:", " ".join(count_backtracking_calls(trap_tickets(3), True)[0]))

 k |   E | số lần gọi dfs | k!
 3 |  10 |             58 | 6
 4 |  12 |            207 | 24
 5 |  14 |            992 | 120
 6 |  16 |          5,887 | 720
 7 |  18 |         41,118 | 5,040


 8 |  20 |        328,823 | 40,320
hành trình với k=3: JFK CCC JFK AAA PAA AAA PBA AAA PCA AAA ZZZ


### 4.6 Nhận xét về Cách 2 → Mấu chốt cho Cách 3

- **Thời gian**: tốt thì gần `O(E log E)` (sort + đi thẳng một mạch như Example 2), **xấu** thì theo hàm mũ / giai thừa như bảng trên. Với `E = 300` và input bẫy, không thể chạy xong. Trên LeetCode, lời giải này có thể qua nhiều test nhưng không có bảo đảm nào — đừng dựa vào nó khi phỏng vấn.
- **Bộ nhớ**: `O(E)` cho `adj`, `used`, `route` và độ sâu đệ quy.

**Nó vướng ở đâu?** Mọi công sức bị phí là do nó **đi vào ngõ cụt quá sớm** (`AAA ... ZZZ`) rồi phải vứt bỏ cả quãng đường đã đi. Nhưng để ý: quãng đường dẫn tới ngõ cụt đó **không hề sai** — nó chỉ **đặt sai chỗ**! Trong đáp án đúng, đoạn `AAA ... ZZZ` vẫn xuất hiện, chỉ là nằm ở **cuối** hành trình.

**Mấu chốt**: thay vì vứt bỏ quãng đường tới ngõ cụt, ta **giữ lại nó làm phần đuôi** của hành trình, rồi chèn các vòng còn sót vào **trước** nó. Ý tưởng này có tên: **thuật toán Hierholzer** tìm **đường đi Euler**. Trước khi học nó, cần hai viên gạch nền: đường đi Euler (mục 5) và post-order (mục 6).

## 5. Nền tảng: đường đi Euler

### 5.1 Đường đi Euler là gì?

Năm 1736, nhà toán học Euler giải câu đố **7 cây cầu ở Königsberg**: *có cách nào đi dạo qua **mỗi cây cầu đúng một lần** không?* Từ đó sinh ra khái niệm:

- **Đường đi Euler** (Euler path / Eulerian trail): một đường đi trong đồ thị dùng **mỗi cạnh đúng một lần** (đỉnh thì được ghé lại tùy ý).
- **Chu trình Euler** (Euler circuit): đường đi Euler mà điểm cuối trùng điểm đầu.

Đọc lại đề 332: *"dùng tất cả vé, mỗi vé đúng một lần, bắt đầu từ JFK"* — đó chính xác là **tìm một đường đi Euler xuất phát từ `JFK`** trong đồ thị có hướng mà mỗi vé là một cạnh! Đề còn đòi thêm: trong các đường Euler, lấy cái nhỏ nhất theo từ điển.

### 5.2 Điều kiện bậc (đồ thị có hướng)

Mỗi lần **đi qua** một sân bay ở giữa hành trình, ta dùng **1 vé vào** và **1 vé ra**. Vì vậy, nếu đồ thị có đường đi Euler từ `S` tới `T` (`S != T`) thì:

- Tại `S` (điểm đầu): `bậc ra - bậc vào = 1` (vé ra đầu tiên không có vé vào đi cặp).
- Tại `T` (điểm cuối): `bậc vào - bậc ra = 1`.
- Mọi sân bay khác: `bậc vào = bậc ra`.

Nếu là chu trình (`S == T`) thì **mọi** sân bay có `bậc vào = bậc ra`. Ngoài ra, các cạnh phải "dính liền" với nhau (đi được từ `S`). Đề đã hứa luôn có hành trình, nên những điều kiện này được bảo đảm sẵn.

In [15]:
def degree_table(tickets):
    out_deg, in_deg = Counter(), Counter()
    for a, b in tickets:
        out_deg[a] += 1
        in_deg[b] += 1
    airports = sorted(set(out_deg) | set(in_deg))
    return [(x, out_deg[x], in_deg[x], out_deg[x] - in_deg[x]) for x in airports]


for name, t in [("Ex1", ex1), ("Ex2", ex2), ("ex3", ex3)]:
    print(name, " (sân bay, bậc ra, bậc vào, ra - vào):")
    for row in degree_table(t):
        tag = "  <- điểm đầu" if row[3] == 1 else ("  <- điểm cuối" if row[3] == -1 else "")
        print("   ", row, tag)

Ex1  (sân bay, bậc ra, bậc vào, ra - vào):
    ('JFK', 1, 0, 1)   <- điểm đầu
    ('LHR', 1, 1, 0) 
    ('MUC', 1, 1, 0) 
    ('SFO', 1, 1, 0) 
    ('SJC', 0, 1, -1)   <- điểm cuối
Ex2  (sân bay, bậc ra, bậc vào, ra - vào):
    ('ATL', 2, 2, 0) 
    ('JFK', 2, 1, 1)   <- điểm đầu
    ('SFO', 1, 2, -1)   <- điểm cuối
ex3  (sân bay, bậc ra, bậc vào, ra - vào):
    ('JFK', 2, 1, 1)   <- điểm đầu
    ('KUL', 0, 1, -1)   <- điểm cuối
    ('NRT', 1, 1, 0) 


### 5.3 Hệ quả quan trọng: chỉ có thể **kẹt** ở điểm cuối

Giả sử ta đi lang thang từ `JFK`, cứ có vé là đi, **không bao giờ quay lui**. Ta có thể kẹt ở đâu?

- Ở một sân bay `x` **không phải** điểm đầu, cũng không phải điểm cuối: `bậc vào = bậc ra`. Mỗi lần **bay vào** `x` ta dùng 1 vé vào; số vé vào đã dùng luôn **nhiều hơn** số vé ra đã dùng đúng 1 (lúc vừa đến). Vì tổng vé ra = tổng vé vào, nên **vẫn còn ít nhất 1 vé ra** chưa dùng → không kẹt được.
- Ở `JFK` (điểm đầu, `bậc ra = bậc vào + 1`): càng không kẹt được, trừ khi JFK là điểm cuối của chu trình (khi đó nó đóng vai trò `T` bên dưới).
- Chỉ ở **điểm cuối `T`** (`bậc vào = bậc ra + 1`) là có thể đến nơi mà hết vé ra.

Kết luận: **lần đầu tiên bị kẹt, ta chắc chắn đang đứng ở điểm cuối `T` của hành trình.** Quãng đường vừa đi có thể chưa dùng hết vé (còn những **vòng** bị bỏ sót dọc đường), nhưng **chỗ kẹt chính là đích cuối cùng**. Ở `ex3`: tham lam kẹt ở `KUL` — và `KUL` đúng là sân bay cuối của đáp án `JFK NRT JFK KUL`.

Còn những vé bị bỏ sót thì sao? Sau khi bỏ đi quãng đường đã đi, ở mọi sân bay số vé vào còn lại = số vé ra còn lại, nên chúng tạo thành **các vòng khép kín** (chu trình), mỗi vòng dính vào một sân bay nào đó trên quãng đường đã đi. Việc còn lại chỉ là **chèn các vòng đó vào đúng chỗ**.

## 6. Nền tảng: pre-order và post-order

Trong DFS đệ quy, ta có thể "ghi lại" một đỉnh ở **hai thời điểm**:

- **Pre-order** (ghi trước): ghi `u` **ngay khi vừa vào** `u`, trước khi đi tiếp. Đây là thứ tự "đi tới đâu ghi tới đó".
- **Post-order** (ghi sau): ghi `u` **khi đã đi xong hết** mọi nhánh từ `u` và chuẩn bị quay về. Đỉnh nào "xong việc" trước thì được ghi trước.

Hierholzer dùng **post-order**: một sân bay chỉ được ghi khi nó **hết vé để bay đi**. Sân bay hết vé **sớm nhất** là ngõ cụt cuối cùng `T` — nên nó được ghi **đầu tiên**, và sau khi **đảo ngược** danh sách, nó nằm **ở cuối** hành trình. Đó là ý nghĩa của câu *"ngõ cụt được thêm vào sau"*: trong danh sách post-order nó vào đầu, trong hành trình thật (đã đảo) nó đứng sau cùng.

Thử trên cây nhỏ của mục 2.5:

In [16]:
def pre_and_post(u, pre, post):
    pre.append(u)                       # ghi khi VÀO
    for v in sorted(tree.get(u, [])):
        pre_and_post(v, pre, post)
    post.append(u)                      # ghi khi RA (đã xong mọi nhánh)


pre, post = [], []
pre_and_post("JFK", pre, post)
print("pre-order          :", pre)
print("post-order         :", post)
print("post-order đảo lại :", post[::-1])

pre-order          : ['JFK', 'ATL', 'BOS', 'LAX', 'SFO', 'SEA']
post-order         : ['BOS', 'LAX', 'ATL', 'SEA', 'SFO', 'JFK']
post-order đảo lại : ['JFK', 'SFO', 'SEA', 'ATL', 'LAX', 'BOS']


Để ý: trong `post-order`, mỗi đỉnh luôn đứng **sau** mọi đỉnh nằm "phía dưới" nó (con cháu của nó). Đảo ngược lại thì mỗi đỉnh đứng **trước** mọi đỉnh nằm phía dưới nó. Tính chất "đảo ngược post-order thì cha đứng trước con" này cũng chính là cách làm **topological sort** bằng DFS (bài *Course Schedule II*) — một mẹo dùng đi dùng lại rất nhiều.

Với bài của ta, "phía dưới" nghĩa là "đi tiếp sau đó". Đảo ngược post-order cho ta một thứ tự mà **mỗi sân bay đứng trước những sân bay đi tiếp sau nó** — đúng thứ ta cần cho một hành trình.

## 7. Cách 3 (cách chính) — Thuật toán Hierholzer, `O(E log E)`

### 7.1 Ý tưởng bằng lời

```text
dfs(u):
    while u còn vé chưa dùng:
        lấy vé u -> v có v NHỎ NHẤT, xé vé (xóa khỏi adj[u])
        dfs(v)
    route.append(u)          # u đã hết vé: ghi u (post-order)

dfs("JFK")
trả về route đảo ngược
```

So với backtracking, có ba điểm khác biệt then chốt:

1. **Không bao giờ trả vé.** Vé đã dùng là xé luôn (`pop`). Không có nhánh nào bị vứt bỏ → mỗi vé được xử lý **đúng một lần**.
2. **Dùng `while`, không phải `for ... return`.** Khi `dfs(v)` quay về `u` mà `u` vẫn còn vé, ta đi tiếp vé tiếp theo — đó chính là lúc **chèn một vòng bị sót** vào hành trình.
3. **Ghi post-order rồi đảo ngược.** Đoạn đường dẫn vào ngõ cụt được ghi **trước** (nên nằm **sau** trong đáp án), các vòng đi sau được ghi **sau** (nên nằm **trước** trong đáp án).

Danh sách kề sắp xếp **giảm dần** để `adj[u].pop()` lấy ra nơi đến nhỏ nhất với chi phí `O(1)` (mục 2.4).

In [17]:
def itinerary_hierholzer(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets, reverse=True):   # giảm dần -> nhỏ nhất ở cuối list
        adj[a].append(b)
    route = []

    def dfs(u):
        while adj[u]:
            v = adj[u].pop()                     # xé vé u->v nhỏ nhất còn lại
            dfs(v)
        route.append(u)                          # u hết vé: ghi theo post-order

    dfs("JFK")
    return route[::-1]


for name, t, ans in [("Ex1", ex1, ans1), ("Ex2", ex2, ans2), ("Ex3", ex3, ans3)]:
    got = itinerary_hierholzer(t)
    print(name, got, "OK" if got == ans else "SAI")
print("Bẫy k=8 :", is_valid_itinerary(trap_tickets(8), itinerary_hierholzer(trap_tickets(8))))

Ex1 ['JFK', 'MUC', 'LHR', 'SFO', 'SJC'] OK
Ex2 ['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO'] OK
Ex3 ['JFK', 'NRT', 'JFK', 'KUL'] OK
Bẫy k=8 : True


### 7.2 Trace Hierholzer trên Example 2

`adj` giảm dần: `JFK: [SFO, ATL]`, `ATL: [SFO, JFK]`, `SFO: [ATL]` (`pop()` lấy phần tử cuối = nhỏ nhất).

| Bước | Sự kiện | `adj` của sân bay đó sau bước | `route` (post-order) |
|---:|---|---|---|
| 1 | vào `JFK`, xé vé `JFK->ATL` | `JFK: [SFO]` | `[]` |
| 2 | vào `ATL`, xé vé `ATL->JFK` | `ATL: [SFO]` | `[]` |
| 3 | vào `JFK`, xé vé `JFK->SFO` | `JFK: []` | `[]` |
| 4 | vào `SFO`, xé vé `SFO->ATL` | `SFO: []` | `[]` |
| 5 | vào `ATL`, xé vé `ATL->SFO` | `ATL: []` | `[]` |
| 6 | vào `SFO`: hết vé → ghi `SFO` | | `[SFO]` |
| 7 | về `ATL`: hết vé → ghi `ATL` | | `[SFO, ATL]` |
| 8 | về `SFO`: hết vé → ghi `SFO` | | `[SFO, ATL, SFO]` |
| 9 | về `JFK`: hết vé → ghi `JFK` | | `[SFO, ATL, SFO, JFK]` |
| 10 | về `ATL`: hết vé → ghi `ATL` | | `[SFO, ATL, SFO, JFK, ATL]` |
| 11 | về `JFK`: hết vé → ghi `JFK` | | `[SFO, ATL, SFO, JFK, ATL, JFK]` |

Đảo ngược: `JFK ATL JFK SFO ATL SFO` — đúng đáp án. Ở ví dụ này cú đi tham lam ở bước 1–5 dùng hết vé ngay, không có vòng nào bị sót; post-order chỉ đơn giản là hành trình viết ngược.

In [18]:
def trace_hierholzer(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets, reverse=True):
        adj[a].append(b)
    route = []

    def dfs(u, depth):
        pad = "  " * depth
        print(f"{pad}vào {u}, vé còn: {adj[u]}")
        while adj[u]:
            v = adj[u].pop()
            print(f"{pad}  xé vé {u}->{v}")
            dfs(v, depth + 1)
        route.append(u)
        print(f"{pad}{u} hết vé -> ghi {u}   route = {route}")

    dfs("JFK", 0)
    print("Đảo ngược:", route[::-1])
    return route[::-1]


trace_hierholzer(ex2);

vào JFK, vé còn: ['SFO', 'ATL']
  xé vé JFK->ATL
  vào ATL, vé còn: ['SFO', 'JFK']
    xé vé ATL->JFK
    vào JFK, vé còn: ['SFO']
      xé vé JFK->SFO
      vào SFO, vé còn: ['ATL']
        xé vé SFO->ATL
        vào ATL, vé còn: ['SFO']
          xé vé ATL->SFO
          vào SFO, vé còn: []
          SFO hết vé -> ghi SFO   route = ['SFO']
        ATL hết vé -> ghi ATL   route = ['SFO', 'ATL']
      SFO hết vé -> ghi SFO   route = ['SFO', 'ATL', 'SFO']
    JFK hết vé -> ghi JFK   route = ['SFO', 'ATL', 'SFO', 'JFK']
  ATL hết vé -> ghi ATL   route = ['SFO', 'ATL', 'SFO', 'JFK', 'ATL']
JFK hết vé -> ghi JFK   route = ['SFO', 'ATL', 'SFO', 'JFK', 'ATL', 'JFK']
Đảo ngược: ['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO']


### 7.3 Trace trên `ex3` — nhìn thấy "ngõ cụt được thêm vào sau"

`adj` giảm dần: `JFK: [NRT, KUL]`, `NRT: [JFK]`.

| Bước | Sự kiện | `route` (post-order) | Giải thích |
|---:|---|---|---|
| 1 | vào `JFK`, xé vé `JFK->KUL` (nhỏ nhất) | `[]` | giống hệt tham lam |
| 2 | vào `KUL`: hết vé → ghi `KUL` | `[KUL]` | **kẹt lần đầu = điểm cuối** (mục 5.3). Ghi ngay, không vứt bỏ! |
| 3 | về `JFK`: **vẫn còn vé** `JFK->NRT` → xé | `[KUL]` | `while` cho phép đi tiếp: bắt đầu một **vòng bị sót** |
| 4 | vào `NRT`, xé vé `NRT->JFK` | `[KUL]` | |
| 5 | vào `JFK`: hết vé → ghi `JFK` | `[KUL, JFK]` | vòng `JFK->NRT->JFK` khép lại |
| 6 | về `NRT`: hết vé → ghi `NRT` | `[KUL, JFK, NRT]` | |
| 7 | về `JFK` (lần gọi ngoài cùng): hết vé → ghi `JFK` | `[KUL, JFK, NRT, JFK]` | |

Đảo ngược: `JFK NRT JFK KUL`. Ngõ cụt `KUL` được ghi **đầu tiên** → đứng **cuối cùng**; vòng `NRT` được ghi **sau** → đứng **trước** nó. Đây là điều backtracking phải "lùi rồi đi lại" mới làm được, còn Hierholzer làm được trong **một lượt**, không phí vé nào.

In [19]:
trace_hierholzer(ex3);

vào JFK, vé còn: ['NRT', 'KUL']
  xé vé JFK->KUL
  vào KUL, vé còn: []
  KUL hết vé -> ghi KUL   route = ['KUL']
  xé vé JFK->NRT
  vào NRT, vé còn: ['JFK']
    xé vé NRT->JFK
    vào JFK, vé còn: []
    JFK hết vé -> ghi JFK   route = ['KUL', 'JFK']
  NRT hết vé -> ghi NRT   route = ['KUL', 'JFK', 'NRT']
JFK hết vé -> ghi JFK   route = ['KUL', 'JFK', 'NRT', 'JFK']
Đảo ngược: ['JFK', 'NRT', 'JFK', 'KUL']


### 7.4 Trace với vòng bị sót nằm **giữa** hành trình

Ở `ex3`, vòng bị sót được chèn ngay sau `JFK` đầu tiên. Để thấy việc "chèn" diễn ra ở giữa, xét `ex4`:

```text
JFK -> ATL,  ATL -> BOS,  ATL -> CDG,  CDG -> ATL
```

Tham lam đi `JFK ATL BOS` rồi kẹt ở `BOS` (ngõ cụt, cũng là điểm cuối), bỏ sót vòng `ATL -> CDG -> ATL`. Hierholzer:

1. Đi `JFK -> ATL -> BOS`, kẹt → ghi `BOS`.
2. Lùi về `ATL`: còn vé `ATL->CDG` → đi vòng `ATL -> CDG -> ATL`. Ở `ATL` lần hai hết vé → ghi `ATL`, rồi ghi `CDG`.
3. Lùi về `ATL` lần một: hết vé → ghi `ATL`. Lùi về `JFK` → ghi `JFK`.

`route = [BOS, ATL, CDG, ATL, JFK]`, đảo lại: `JFK ATL CDG ATL BOS`. Vòng `CDG` đã được **chèn vào giữa** `ATL` và `BOS` — đúng vị trí nó dính vào quãng đường chính.

In [20]:
ex4 = [["JFK", "ATL"], ["ATL", "BOS"], ["ATL", "CDG"], ["CDG", "ATL"]]
ans4 = ["JFK", "ATL", "CDG", "ATL", "BOS"]
r4 = trace_hierholzer(ex4)
print("Đúng đáp án:", r4 == ans4, "| tham lam cho:", itinerary_greedy_wrong(ex4))

vào JFK, vé còn: ['ATL']
  xé vé JFK->ATL
  vào ATL, vé còn: ['CDG', 'BOS']
    xé vé ATL->BOS
    vào BOS, vé còn: []
    BOS hết vé -> ghi BOS   route = ['BOS']
    xé vé ATL->CDG
    vào CDG, vé còn: ['ATL']
      xé vé CDG->ATL
      vào ATL, vé còn: []
      ATL hết vé -> ghi ATL   route = ['BOS', 'ATL']
    CDG hết vé -> ghi CDG   route = ['BOS', 'ATL', 'CDG']
  ATL hết vé -> ghi ATL   route = ['BOS', 'ATL', 'CDG', 'ATL']
JFK hết vé -> ghi JFK   route = ['BOS', 'ATL', 'CDG', 'ATL', 'JFK']
Đảo ngược: ['JFK', 'ATL', 'CDG', 'ATL', 'BOS']
Đúng đáp án: True | tham lam cho: ['JFK', 'ATL', 'BOS']


## 8. Cách 4 — Hierholzer bằng stack (không đệ quy)

### 8.1 Vì sao cần bản lặp?

Bản đệ quy có độ sâu tối đa `E + 1` (ví dụ một chuỗi thẳng 300 vé). Với `E <= 300`, giới hạn đệ quy mặc định của Python (1000) là đủ. Nhưng ở các bài anh em (ví dụ *Valid Arrangement of Pairs* có tới `10^5` cạnh), đệ quy sẽ lỗi `RecursionError`. Hơn nữa, viết bằng stack giúp **nhìn rõ** cái call stack mà đệ quy đang giấu.

**Stack** (ngăn xếp) là cấu trúc "vào sau, ra trước" (LIFO): `append` để đẩy vào đỉnh, `pop()` để lấy ra từ đỉnh, `stack[-1]` để xem đỉnh. List Python làm stack rất tốt.

### 8.2 Chuyển đệ quy thành vòng lặp

Đỉnh stack đóng vai "sân bay đang đứng trong lần gọi `dfs` sâu nhất":

```text
stack = ["JFK"], route = []
while stack:
    u = stack[-1]                     # đang đứng ở u
    if adj[u]:                        # u còn vé  <=> trong dfs(u), vòng while chạy tiếp
        stack.append(adj[u].pop())    #            <=> gọi dfs(v)
    else:                             # u hết vé  <=> dfs(u) kết thúc
        route.append(stack.pop())     #            <=> route.append(u) rồi return
return route[::-1]
```

Mỗi dòng tương ứng 1–1 với bản đệ quy, nên kết quả **giống hệt**.

In [21]:
def itinerary_stack(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets, reverse=True):
        adj[a].append(b)
    stack, route = ["JFK"], []
    while stack:
        u = stack[-1]
        if adj[u]:
            stack.append(adj[u].pop())      # còn vé: bay tiếp tới nơi nhỏ nhất
        else:
            route.append(stack.pop())       # hết vé: ghi u (post-order)
    return route[::-1]


for name, t, ans in [("Ex1", ex1, ans1), ("Ex2", ex2, ans2), ("Ex3", ex3, ans3), ("ex4", ex4, ans4)]:
    got = itinerary_stack(t)
    print(name, got, "OK" if got == ans else "SAI")

Ex1 ['JFK', 'MUC', 'LHR', 'SFO', 'SJC'] OK
Ex2 ['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO'] OK
Ex3 ['JFK', 'NRT', 'JFK', 'KUL'] OK
ex4 ['JFK', 'ATL', 'CDG', 'ATL', 'BOS'] OK


### 8.3 Trace bản stack trên `ex4`

`adj` giảm dần: `JFK: [ATL]`, `ATL: [CDG, BOS]`, `CDG: [ATL]`.

| Vòng | `stack` trước | đỉnh `u` | `u` còn vé? | Hành động | `route` sau |
|---:|---|---|:---:|---|---|
| 1 | `[JFK]` | JFK | có | xé `JFK->ATL`, đẩy ATL | `[]` |
| 2 | `[JFK, ATL]` | ATL | có | xé `ATL->BOS`, đẩy BOS | `[]` |
| 3 | `[JFK, ATL, BOS]` | BOS | không | lấy BOS ra, ghi | `[BOS]` |
| 4 | `[JFK, ATL]` | ATL | có | xé `ATL->CDG`, đẩy CDG | `[BOS]` |
| 5 | `[JFK, ATL, CDG]` | CDG | có | xé `CDG->ATL`, đẩy ATL | `[BOS]` |
| 6 | `[JFK, ATL, CDG, ATL]` | ATL | không | lấy ATL ra, ghi | `[BOS, ATL]` |
| 7 | `[JFK, ATL, CDG]` | CDG | không | lấy CDG ra, ghi | `[BOS, ATL, CDG]` |
| 8 | `[JFK, ATL]` | ATL | không | lấy ATL ra, ghi | `[BOS, ATL, CDG, ATL]` |
| 9 | `[JFK]` | JFK | không | lấy JFK ra, ghi | `[BOS, ATL, CDG, ATL, JFK]` |

Vòng 3 là khoảnh khắc "kẹt lần đầu" ở ngõ cụt `BOS`; vòng 4 là lúc `ATL` "nhớ ra" còn vé và mở một vòng phụ. Tổng số vòng lặp = `2E + 1` = 9: mỗi vé gây ra đúng một lần đẩy, mỗi sân bay trong hành trình gây ra đúng một lần lấy ra.

In [22]:
def trace_stack(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets, reverse=True):
        adj[a].append(b)
    stack, route = ["JFK"], []
    step = 0
    while stack:
        step += 1
        u = stack[-1]
        before = list(stack)
        if adj[u]:
            v = adj[u].pop()
            stack.append(v)
            action = f"xé {u}->{v}, đẩy {v}"
        else:
            route.append(stack.pop())
            action = f"lấy {u} ra, ghi"
        print(f"{step}: stack={before!s:<28} u={u}  {action:<20} route={route}")
    return route[::-1]


print("Kết quả:", trace_stack(ex4))

1: stack=['JFK']                      u=JFK  xé JFK->ATL, đẩy ATL route=[]
2: stack=['JFK', 'ATL']               u=ATL  xé ATL->BOS, đẩy BOS route=[]
3: stack=['JFK', 'ATL', 'BOS']        u=BOS  lấy BOS ra, ghi      route=['BOS']
4: stack=['JFK', 'ATL']               u=ATL  xé ATL->CDG, đẩy CDG route=['BOS']
5: stack=['JFK', 'ATL', 'CDG']        u=CDG  xé CDG->ATL, đẩy ATL route=['BOS']
6: stack=['JFK', 'ATL', 'CDG', 'ATL'] u=ATL  lấy ATL ra, ghi      route=['BOS', 'ATL']
7: stack=['JFK', 'ATL', 'CDG']        u=CDG  lấy CDG ra, ghi      route=['BOS', 'ATL', 'CDG']
8: stack=['JFK', 'ATL']               u=ATL  lấy ATL ra, ghi      route=['BOS', 'ATL', 'CDG', 'ATL']
9: stack=['JFK']                      u=JFK  lấy JFK ra, ghi      route=['BOS', 'ATL', 'CDG', 'ATL', 'JFK']
Kết quả: ['JFK', 'ATL', 'CDG', 'ATL', 'BOS']


## 9. Code nộp LeetCode

Chuyển `itinerary_hierholzer` thành đúng format LeetCode: method `findItinerary` của class `Solution`, nhận `tickets: List[List[str]]`, trả `List[str]`. Không `print`, không biến toàn cục.

- Sắp xếp vé **giảm dần** một lần, dựng `adj` bằng `defaultdict(list)`.
- `dfs` lồng bên trong method để dùng chung `adj` và `route`.
- Trả `route[::-1]`.

Độ sâu đệ quy tối đa `E + 1 <= 301`, an toàn với giới hạn mặc định 1000 của Python.

In [23]:
class Solution:
    def findItinerary(self, tickets: List[List[str]]) -> List[str]:
        adj = defaultdict(list)
        for a, b in sorted(tickets, reverse=True):
            adj[a].append(b)
        route = []

        def dfs(u: str) -> None:
            while adj[u]:
                dfs(adj[u].pop())
            route.append(u)

        dfs("JFK")
        return route[::-1]


sol = Solution()
assert sol.findItinerary(ex1) == ans1
assert sol.findItinerary(ex2) == ans2
assert sol.findItinerary(ex3) == ans3
assert sol.findItinerary(ex4) == ans4
assert sol.findItinerary([["JFK", "AAA"]]) == ["JFK", "AAA"]                      # 1 vé
assert sol.findItinerary([["JFK", "ATL"], ["ATL", "JFK"]]) == ["JFK", "ATL", "JFK"]  # chu trình
assert sol.findItinerary(dup) == ["JFK", "BBB", "JFK", "BBB", "JFK", "BBB", "JFK", "AAA"]  # vé trùng
chain = [[f"A{i:02d}" if i else "JFK", f"A{i + 1:02d}"] for i in range(300)]          # chuỗi thẳng 300 vé
random.seed(1); random.shuffle(chain)
assert len(sol.findItinerary(chain)) == 301 and is_valid_itinerary(chain, sol.findItinerary(chain))
print("Solution (Hierholzer đệ quy): tất cả test PASS")

Solution (Hierholzer đệ quy): tất cả test PASS


In [24]:
# Bản KHÔNG đệ quy (Cách 4). Khi nộp LeetCode thì đổi tên SolutionStack -> Solution.
class SolutionStack:
    def findItinerary(self, tickets: List[List[str]]) -> List[str]:
        adj = defaultdict(list)
        for a, b in sorted(tickets, reverse=True):
            adj[a].append(b)
        stack, route = ["JFK"], []
        while stack:
            if adj[stack[-1]]:
                stack.append(adj[stack[-1]].pop())
            else:
                route.append(stack.pop())
        return route[::-1]


sol2 = SolutionStack()
for t, ans in [(ex1, ans1), (ex2, ans2), (ex3, ans3), (ex4, ans4)]:
    assert sol2.findItinerary(t) == ans
assert sol2.findItinerary(chain) == sol.findItinerary(chain)
print("SolutionStack: tất cả test PASS")

SolutionStack: tất cả test PASS


## 10. Đối chiếu chéo các cách trên input ngẫu nhiên

Làm sao sinh ngẫu nhiên một bộ vé **chắc chắn có hành trình**? Mẹo: cho một người **đi dạo ngẫu nhiên** từ `JFK` qua `E` chặng (mỗi chặng tới một sân bay khác sân bay hiện tại), ghi lại từng chặng làm vé, rồi **xáo trộn** thứ tự vé. Chính cuộc dạo đó là một hành trình hợp lệ, nên đề bài luôn thỏa.

Dùng ít sân bay (`JFK, ATL, SFO, LHR`) để tạo nhiều vé trùng và nhiều vòng — đúng loại input dễ làm lộ lỗi. Brute force chỉ chạy được khi `E <= 6`; các cách còn lại chạy tới `E = 12` và một test lớn `E = 300`.

In [25]:
AIRPORTS = ["JFK", "ATL", "SFO", "LHR"]


def random_tickets(E):
    cur, t = "JFK", []
    for _ in range(E):
        nxt = random.choice([x for x in AIRPORTS if x != cur])
        t.append([cur, nxt])
        cur = nxt
    random.shuffle(t)
    return t


random.seed(332)
for _ in range(300):
    t = random_tickets(random.randint(1, 6))
    expected = itinerary_bruteforce([list(x) for x in t])
    for f in (itinerary_backtracking, itinerary_hierholzer, itinerary_stack,
              Solution().findItinerary, SolutionStack().findItinerary):
        assert f([list(x) for x in t]) == expected, (f, t)
print("E <= 6 : 5 cách khớp brute force trên 300 bộ vé")

for _ in range(300):
    t = random_tickets(random.randint(7, 12))
    expected = itinerary_backtracking([list(x) for x in t])
    assert is_valid_itinerary(t, expected)
    assert itinerary_hierholzer([list(x) for x in t]) == expected
    assert itinerary_stack([list(x) for x in t]) == expected
print("E <= 12: Hierholzer (2 bản) khớp backtracking trên 300 bộ vé")

big_t = random_tickets(300)
t0 = time.perf_counter(); r = Solution().findItinerary([list(x) for x in big_t]); t1 = time.perf_counter()
print(f"E = 300: hợp lệ = {is_valid_itinerary(big_t, r)}, thời gian Hierholzer ≈ {(t1 - t0) * 1000:.2f} ms")

E <= 6 : 5 cách khớp brute force trên 300 bộ vé
E <= 12: Hierholzer (2 bản) khớp backtracking trên 300 bộ vé
E = 300: hợp lệ = True, thời gian Hierholzer ≈ 0.36 ms


## 11. Vì sao các cách đúng?

**Cách 1 (brute force)**: thử **mọi** thứ tự vé nên không bỏ sót hành trình nào; giữ `min` nên trả đúng cái nhỏ nhất. Đúng nhưng chậm.

**Cách 2 (backtracking)**: chỉ khác brute force ở chỗ *cắt sớm* các nhánh không nối được (không làm mất hành trình hợp lệ nào) và *thử theo thứ tự từ điển*. Mục 4.2 đã chỉ ra: vì mọi hành trình dài bằng nhau, hành trình hoàn chỉnh **đầu tiên** DFS gặp được là nhỏ nhất.

**Cách 3/4 (Hierholzer)** — ba ý:

1. **Mỗi vé dùng đúng một lần.** Mỗi vé bị `pop` đúng một lần và không bao giờ được trả lại. Mỗi lần `pop` dẫn tới đúng một sân bay được ghi vào `route` (khi lần gọi đó kết thúc), cộng thêm `JFK` của lần gọi ngoài cùng → `route` có đúng `E + 1` phần tử.
2. **`route` đảo ngược là một hành trình hợp lệ.** *Invariant*: khi một sân bay `u` được ghi vào `route`, phần đã ghi trước nó (đọc ngược) là một đoạn hành trình **bắt đầu từ chỗ `u` vừa bay tới**. Lý do: `u` chỉ được ghi khi đã hết vé, và thứ được ghi ngay trước nó chính là điểm kết thúc của lần gọi `dfs(v)` cuối cùng mà `u` thực hiện — mà `dfs(v)` bắt đầu bằng vé `u -> v`. Ghép dần các đoạn ta được một chuỗi liền mạch từ `JFK` tới ngõ cụt.
3. **Nhỏ nhất theo từ điển.** Ở mỗi sân bay ta luôn đi vé nhỏ nhất còn lại trước. Vé nhỏ hơn chỉ bị "đẩy ra sau" khi nhánh của nó **kẹt** ở điểm cuối `T` mà chưa quay lại được (mục 5.3). Mà một hành trình hợp lệ thì **không thể** đi vào điểm cuối khi còn vé chưa dùng, nên **mọi** hành trình hợp lệ đều phải đi các vòng còn sót **trước** đoạn kết thúc ở `T`. Hierholzer chèn các vòng đó vào đúng chỗ sớm nhất có thể, và bên trong mỗi vòng lại đi vé nhỏ nhất trước. Chứng minh chặt chẽ khá dài; ở đây ta củng cố thêm bằng kiểm chứng: Hierholzer khớp brute force trên hàng trăm bộ vé ngẫu nhiên (mục 10).

## 12. Độ phức tạp và so sánh

Ký hiệu: `E` = số vé, `V` = số sân bay (`V <= E + 1`).

| Cách | Thời gian | Bộ nhớ thêm | Ghi chú |
|---|---|---|---|
| 1. Brute force hoán vị | `O(E! · E)` | `O(E)` | chỉ để hiểu đề, `E <= 8` là hết sức |
| 2. Backtracking theo từ điển | tốt: `O(E log E)`; xấu: hàm mũ (thấy `k!` ở mục 4.5) | `O(E)` | dễ nghĩ, dừng ở lời giải đầu tiên, không có bảo đảm tốc độ |
| 3. Hierholzer đệ quy | `O(E log E)` | `O(V + E)` + đệ quy sâu `O(E)` | **cách chính** |
| 4. Hierholzer bằng stack | `O(E log E)` | `O(V + E)` | giống Cách 3, không lo giới hạn đệ quy |

Vì sao `O(E log E)` cho Hierholzer: sort `E` vé tốn `O(E log E)`; sau đó mỗi vé `pop` đúng một lần (`O(1)` mỗi lần), mỗi sân bay của hành trình được ghi một lần, đảo ngược `O(E)`. Phần sort là phần đắt nhất. Bộ nhớ: `adj` chứa `E` nơi đến trên `V` key, `route` dài `E + 1`.

## 13. Lỗi thường gặp

### Lỗi 1: Tham lam không quay lui

Đã thấy ở mục 4.1: `itinerary_greedy_wrong(ex3)` trả `['JFK', 'KUL']`, mới dùng 1/3 vé. **Sửa**: hoặc backtracking (trả vé khi kẹt), hoặc Hierholzer (giữ đoạn kẹt làm đuôi, dùng `while` để đi tiếp các vé còn lại và ghi theo post-order).

### Lỗi 2: Ghi sân bay theo **pre-order** (ghi ngay khi vào)

Nhiều bạn viết `route.append(u)` ở **đầu** hàm `dfs` vì thấy "tự nhiên" hơn, và bỏ luôn bước đảo ngược. Với Example 1, 2 vẫn ra đúng (không có vòng bị sót), nên rất khó phát hiện! Nhưng với `ex3`:

In [26]:
def itinerary_preorder_wrong(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets, reverse=True):
        adj[a].append(b)
    route = []

    def dfs(u):
        route.append(u)              # SAI: ghi khi vừa vào
        while adj[u]:
            dfs(adj[u].pop())

    dfs("JFK")
    return route


for name, t in [("Ex2", ex2), ("ex3", ex3)]:
    r = itinerary_preorder_wrong(t)
    print(f"{name}: pre-order = {r}, hợp lệ: {is_valid_itinerary(t, r)}")
print("ex3 đúng phải là:", itinerary_hierholzer(ex3))

Ex2: pre-order = ['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO'], hợp lệ: True
ex3: pre-order = ['JFK', 'KUL', 'NRT', 'JFK'], hợp lệ: False
ex3 đúng phải là: ['JFK', 'NRT', 'JFK', 'KUL']


Pre-order ghi `JFK KUL` rồi mới nối thêm `NRT JFK` — nhưng không có vé `KUL -> NRT`! Pre-order ghi theo **thứ tự đi**, mà thứ tự đi của Hierholzer có những "cú nhảy" (sau khi kẹt ở `KUL` ta quay về `JFK` bằng cách lùi trong đệ quy, không phải bằng vé). Chỉ **post-order + đảo ngược** mới đặt ngõ cụt về cuối. **Sửa**: `route.append(u)` đặt **sau** vòng `while`, và trả `route[::-1]`.

### Lỗi 3: Sắp xếp tăng dần rồi dùng `pop()`

`pop()` lấy phần tử **cuối**. Nếu list sắp tăng dần thì phần tử cuối là nơi đến **lớn nhất** → vẫn ra một hành trình **hợp lệ**, nhưng không phải nhỏ nhất. Rất nguy hiểm vì code không lỗi, `is_valid` vẫn `True`!

In [27]:
def itinerary_ascending_pop_wrong(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets):                 # SAI: tăng dần
        adj[a].append(b)
    route = []

    def dfs(u):
        while adj[u]:
            dfs(adj[u].pop())                    # pop() lấy cái LỚN nhất
        route.append(u)

    dfs("JFK")
    return route[::-1]


r = itinerary_ascending_pop_wrong(ex2)
print("tăng dần + pop():", r, "| hợp lệ:", is_valid_itinerary(ex2, r), "| nhỏ nhất:", r == ans2)
print("đáp án đúng     :", ans2)

tăng dần + pop(): ['JFK', 'SFO', 'ATL', 'JFK', 'ATL', 'SFO'] | hợp lệ: True | nhỏ nhất: False
đáp án đúng     : ['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO']


Kết quả chính là "cách dựng khác" mà đề nhắc tới — hợp lệ nhưng lớn hơn. **Sửa**: hoặc sort giảm dần + `pop()`, hoặc sort tăng dần + `pop(0)` (đúng nhưng mỗi lần `O(k)`), hoặc dùng `deque.popleft()`, hoặc dùng heap (mục 14).

### Lỗi 4: Dùng `visited` cho **sân bay**

Thói quen từ các bài BFS/DFS khác: đánh dấu sân bay đã ghé để "khỏi đi lại". Nhưng ở bài này sân bay **được phép** ghé lại (`JFK` xuất hiện 2 lần trong đáp án Example 2). Thứ cần đánh dấu là **vé** (cạnh), không phải sân bay (đỉnh).

In [28]:
def itinerary_visited_airports_wrong(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets):
        adj[a].append(b)
    visited = {"JFK"}
    route = ["JFK"]

    def dfs(u):
        for v in adj[u]:
            if v not in visited:                 # SAI: cấm quay lại sân bay cũ
                visited.add(v)
                route.append(v)
                dfs(v)
                return

    dfs("JFK")
    return route


r = itinerary_visited_airports_wrong(ex2)
print("visited sân bay:", r, f"| dùng {len(r) - 1}/{len(ex2)} vé | hợp lệ:", is_valid_itinerary(ex2, r))

visited sân bay: ['JFK', 'ATL', 'SFO'] | dùng 2/5 vé | hợp lệ: False


**Sửa**: đánh dấu vé — bằng `used[u][i]` (backtracking) hoặc bằng cách **xóa vé khỏi `adj`** khi dùng (Hierholzer).

### Các lỗi nhỏ khác (đặt cạnh nhau cho dễ nhớ)

| Lỗi | Ví dụ sai | Vì sao sai / cách sửa |
|---|---|---|
| Quên đảo ngược | `return route` | `route` là post-order, đọc từ ngõ cụt về `JFK`. Sửa: `return route[::-1]`. |
| Dùng `set` cho danh sách kề | `adj[a].add(b)` | Mất vé trùng (mục 2.2). Sửa: dùng `list`. |
| Dùng `for v in adj[u]` trong Hierholzer | `for v in adj[u]: adj[u].remove(v); dfs(v)` | Sửa list khi đang duyệt nó → bỏ sót phần tử. Sửa: `while adj[u]: v = adj[u].pop()`. |
| Backtracking xóa vé bằng `remove` rồi `append` trả lại | `adj[u].remove(v) ... adj[u].append(v)` | Vé trả lại nằm cuối list → **phá thứ tự từ điển** của các lần thử sau. Sửa: dùng cờ `used[u][i]`, hoặc `insert(i, v)` về đúng chỗ. |
| Backtracking quên điều kiện dừng `len(route) == E + 1` | dừng khi "hết vé tại `u`" | Hết vé tại `u` có thể là ngõ cụt sớm (như `KUL`) chứ chưa dùng hết vé. |
| Bắt đầu từ sân bay nhỏ nhất thay vì `JFK` | `dfs(min(adj))` | Đề bắt buộc xuất phát từ `JFK` (`"ATL" < "JFK"` nhưng không được bắt đầu ở `ATL`). |

## 14. Mẹo nhận dạng pattern, cách cài đặt khác và bài liên quan

### 14.1 Dấu hiệu "đây là bài đường đi Euler"

- Đề nói **"dùng mỗi cạnh / mỗi vé / mỗi cặp / mỗi domino đúng một lần"** (chứ không phải mỗi **đỉnh** một lần — đó là đường Hamilton, khó hơn nhiều, NP-hard).
- Các mảnh phải **nối đuôi** nhau: "đầu của mảnh sau = đuôi của mảnh trước".
- Đề **hứa** luôn có lời giải → thường là đã bảo đảm điều kiện bậc.

Khi thấy các dấu hiệu đó, nghĩ ngay: **Hierholzer = DFS xé cạnh + post-order + đảo ngược**. Nếu đề đòi nhỏ nhất theo từ điển → thêm sắp xếp danh sách kề.

### 14.2 Một cách giữ "nơi đến nhỏ nhất" khác: heap (`heapq`)

Thay vì sort giảm dần rồi `pop()`, ta có thể để mỗi `adj[u]` là một **min-heap** (đống nhỏ nhất). Heap là cấu trúc luôn cho lấy ra phần tử **nhỏ nhất** với chi phí `O(log k)`. Python có module `heapq` làm việc trên list thường:

- `heapq.heapify(h)`: biến list `h` thành heap, `O(k)`.
- `heapq.heappush(h, x)`: thêm `x`, `O(log k)`.
- `heapq.heappop(h)`: lấy ra và xóa phần tử nhỏ nhất, `O(log k)`.
- `h[0]`: xem phần tử nhỏ nhất (không xóa).

Ưu điểm của heap: thêm vé mới vào giữa chừng vẫn giữ được thứ tự (sort một lần thì không). Với bài 332 cả hai đều cho `O(E log E)`. Bài tập 4 sẽ để bạn tự ráp heap vào bản stack.

In [29]:
h = ["SFO", "ATL", "LHR"]
heapq.heapify(h)
print("sau heapify, h[0] =", h[0], "| list bên trong:", h)
heapq.heappush(h, "AAA")
print("push AAA,    h[0] =", h[0])
print("pop lần lượt     :", [heapq.heappop(h) for _ in range(len(h))])

sau heapify, h[0] = ATL | list bên trong: ['ATL', 'SFO', 'LHR']
push AAA,    h[0] = AAA
pop lần lượt     : ['AAA', 'ATL', 'LHR', 'SFO']


### 14.3 Bài liên quan

| Bài | Liên hệ với 332 |
|---|---|
| **2097. Valid Arrangement of Pairs** (Hard) | Gần như y hệt, nhưng không cho sẵn điểm đầu: phải **tìm điểm đầu bằng bậc** (`ra - vào = 1`), và không đòi thứ tự từ điển. `10^5` cạnh → nên dùng bản stack. (Bài tập 5.) |
| **753. Cracking the Safe** (Hard) | Dựng **chuỗi de Bruijn**: mỗi chuỗi `n-1` ký tự là một đỉnh, mỗi mật khẩu `n` ký tự là một cạnh; đáp án là một **chu trình Euler**, giải bằng Hierholzer. |
| **210. Course Schedule II** (Medium) | Cùng mẹo **đảo ngược post-order** để ra thứ tự topological (mục 6). |
| **Đường Euler vô hướng** (bài luyện tập kinh điển, "domino", "7 cây cầu") | Điều kiện bậc đổi thành: số đỉnh bậc lẻ là 0 hoặc 2; khi xé một cạnh vô hướng phải xóa ở **cả hai** đầu. |
| **Khi không bảo đảm có lời giải** | Chạy Hierholzer rồi **kiểm tra lại** kết quả (đủ `E + 1` và mọi bước là một vé). (Bài tập 6.) |

## 15. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. **Tự làm trước**, xem gợi ý ở mục 16 nếu bí, đáp án ở mục 17 (cuối notebook). Các cell TODO dưới chỉ **định nghĩa** hàm (thân là `pass` / `raise NotImplementedError`) và để sẵn test ở dạng **comment** — làm xong thì bỏ comment để tự kiểm tra.

### Bài 1 — Đoán output (không chạy code)

```python
print(sorted(["SFO", "JFK", "ATL", "JFA"]))
print(["JFK", "ATL", "ZZZ"] < ["JFK", "BOS", "AAA"])
adj = defaultdict(list)
for a, b in sorted([["JFK", "SFO"], ["JFK", "ATL"], ["JFK", "MUC"]], reverse=True):
    adj[a].append(b)
print(adj["JFK"], adj["JFK"].pop(), adj["JFK"])
print(len(adj), "ATL" in adj, len(adj["ATL"]), len(adj))
```

Dòng cuối có một cái bẫy liên quan tới `defaultdict` (mục 2.1).

### Bài 2 — Tính tay Hierholzer

Với bộ vé:

```text
tickets = [["JFK","AAA"],["AAA","JFK"],["JFK","BBB"],["JFK","CCC"],["CCC","JFK"]]
```

1. Viết `adj` sau khi sắp xếp giảm dần.
2. Lập bảng giống mục 7.3: mỗi dòng là "vào X, xé vé X->Y" hoặc "X hết vé → ghi X", kèm `route` (post-order) sau dòng đó.
3. Đảo ngược để ra đáp án. Sân bay nào là ngõ cụt? Nó được ghi vào `route` ở lần thứ mấy?

### Bài 3 — Điền TODO: Hierholzer đệ quy

Hoàn thiện hàm theo đúng khuôn của mục 7. Đừng mở lại mục 7 nếu chưa thử ít nhất một lần!

In [30]:
def exercise_find_itinerary(tickets):
    adj = defaultdict(list)
    # TODO 1: dựng adj sao cho adj[u].pop() luôn trả về nơi đến NHỎ NHẤT còn lại
    route = []

    def dfs(u):
        # TODO 2: khi u còn vé: xé vé nhỏ nhất u -> v rồi đi tiếp dfs(v)
        # TODO 3: khi u hết vé: ghi u vào route (post-order)
        pass

    # TODO 4: bắt đầu từ đâu? trả về gì?
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert exercise_find_itinerary(ex1) == ans1
# assert exercise_find_itinerary(ex2) == ans2
# assert exercise_find_itinerary(ex3) == ans3
# assert exercise_find_itinerary(ex4) == ans4

### Bài 4 — Tự viết: bản stack dùng heap

Viết lại Cách 4 (stack, không đệ quy), nhưng thay vì sort giảm dần + `pop()`, hãy để mỗi `adj[u]` là một **min-heap** và dùng `heapq.heappop` để lấy nơi đến nhỏ nhất (mục 14.2).

In [31]:
def exercise_itinerary_heap_stack(tickets):
    # TODO: dựng adj[u] là min-heap (heappush từng vé, hoặc append rồi heapify)
    # TODO: stack = ["JFK"], route = []
    # TODO: while stack: nếu đỉnh còn vé -> heappop và đẩy vào stack; không thì ghi vào route
    # TODO: trả route đảo ngược
    pass


# Tự làm xong rồi mới mở test:
# for t, ans in [(ex1, ans1), (ex2, ans2), (ex3, ans3), (ex4, ans4)]:
#     assert exercise_itinerary_heap_stack(t) == ans

### Bài 5 — Biến thể: LeetCode 2097. Valid Arrangement of Pairs (Hard)

> Cho mảng 2 chiều `pairs` (đánh chỉ số từ 0), `pairs[i] = [start_i, end_i]`. Một cách sắp xếp `pairs` là **hợp lệ** nếu với mọi `1 <= i < pairs.length` ta có `end_{i-1} == start_i`. Trả về **một** cách sắp xếp hợp lệ bất kỳ. Input bảo đảm luôn tồn tại cách sắp xếp hợp lệ.
>
> Ví dụ: `pairs = [[5,1],[4,5],[11,9],[9,4]]` → `[[11,9],[9,4],[4,5],[5,1]]`; `pairs = [[1,3],[3,2],[2,1]]` → `[[1,3],[3,2],[2,1]]`; `pairs = [[1,2],[1,3],[2,1]]` → `[[1,2],[2,1],[1,3]]`.
>
> Constraints: `1 <= pairs.length <= 10^5`, `pairs[i].length == 2`, `0 <= start_i, end_i <= 10^9`, `start_i != end_i`, không có hai cặp nào giống hệt nhau.

Signature LeetCode: `def validArrangement(self, pairs: List[List[int]]) -> List[List[int]]`. Khác bài 332 ở chỗ: **không có `JFK`** — bạn phải tự tìm điểm xuất phát; không cần thứ tự từ điển; và `10^5` cạnh nên **đừng dùng đệ quy**. Vì đề cho phép trả "bất kỳ", test sẽ kiểm tra **tính hợp lệ** chứ không so với một đáp án cố định.

In [32]:
def exercise_valid_arrangement(pairs):
    # TODO 1: tính bậc ra - bậc vào cho mỗi đỉnh; điểm đầu là đỉnh có hiệu = 1,
    #         nếu không có đỉnh nào như vậy (chu trình) thì lấy pairs[0][0]
    # TODO 2: Hierholzer bằng stack từ điểm đầu, ra danh sách đỉnh theo đúng thứ tự đi
    # TODO 3: chuyển dãy đỉnh [x0, x1, ..., xE] thành các cặp [[x0,x1], [x1,x2], ...]
    pass


def is_valid_arrangement(pairs, arranged):
    return (sorted(map(tuple, pairs)) == sorted(map(tuple, arranged))
            and all(arranged[i - 1][1] == arranged[i][0] for i in range(1, len(arranged))))


# Tự làm xong rồi mới mở test:
# for p in [[[5, 1], [4, 5], [11, 9], [9, 4]], [[1, 3], [3, 2], [2, 1]], [[1, 2], [1, 3], [2, 1]]]:
#     assert is_valid_arrangement(p, exercise_valid_arrangement(p))

### Bài 6 — Mini challenge: khi **không** bảo đảm có hành trình

Bỏ lời hứa "luôn có hành trình hợp lệ". Viết `exercise_itinerary_or_empty(tickets)` trả về hành trình nhỏ nhất theo từ điển nếu tồn tại, ngược lại trả `[]`.

Câu hỏi suy nghĩ trước khi code: nếu cứ chạy Hierholzer trên `[["JFK","AAA"],["JFK","BBB"]]` (không có hành trình nào), nó trả ra cái gì? Kết quả đó có "trông giống" một hành trình không? Vì sao chỉ cần **kiểm tra lại** kết quả của Hierholzer là đủ, không cần thuật toán mới?

Kiểm tra bằng cách đối chiếu với brute force trên bộ vé ngẫu nhiên **hoàn toàn** (không sinh từ cuộc dạo), `E <= 6`.

In [33]:
def exercise_itinerary_or_empty(tickets):
    # TODO: chạy Hierholzer như bình thường
    # TODO: nếu kết quả là một hành trình hợp lệ thì trả nó, ngược lại trả []
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert exercise_itinerary_or_empty([["JFK", "AAA"], ["JFK", "BBB"]]) == []
# assert exercise_itinerary_or_empty([["ATL", "SFO"]]) == []
# assert exercise_itinerary_or_empty(ex3) == ans3
# random.seed(6)
# for _ in range(300):
#     t = [random.sample(AIRPORTS, 2) for _ in range(random.randint(1, 6))]
#     assert exercise_itinerary_or_empty(t) == (itinerary_bruteforce(t) or [])

## 16. Gợi ý theo mức

Mỗi bài có 3 mức: **(1)** hướng tư duy, **(2)** cấu trúc / cú pháp, **(3)** gần-code. Chỉ mở mức tiếp theo khi mức trước chưa đủ.

**Bài 1**
1. Chuỗi so từng ký tự; list so từng phần tử; dừng ở chỗ khác nhau **đầu tiên**.
2. `"JFA"` và `"JFK"` khác nhau ở ký tự thứ 3. `pop()` lấy phần tử cuối. Đối số của `print` được tính **từ trái sang phải** trước khi in.
3. `adj["JFK"]` sau khi sort giảm dần là `["SFO", "MUC", "ATL"]`. Câu `len(adj["ATL"])` tạo key mới — vì vậy `len(adj)` ở cuối dòng khác `len(adj)` ở đầu dòng.

**Bài 2**
1. Ngõ cụt là sân bay **không có vé bay đi**. Nó sẽ là sân bay cuối của đáp án.
2. `adj` giảm dần: `JFK: [CCC, BBB, AAA]`, `AAA: [JFK]`, `CCC: [JFK]`. Từ `JFK` bạn sẽ xé `AAA` trước.
3. Sau khi về lại `JFK` từ `AAA`, `JFK` còn `[CCC, BBB]` → xé `BBB` → `BBB` hết vé → ghi `BBB` **đầu tiên**.

**Bài 3**
1. Ba ý: sort giảm dần; `while` chứ không phải `if`; ghi sau vòng lặp.
2. `for a, b in sorted(tickets, reverse=True): adj[a].append(b)`.
3. `while adj[u]: dfs(adj[u].pop())` rồi `route.append(u)`; cuối cùng `dfs("JFK")` và `return route[::-1]`.

**Bài 4**
1. Chỉ cần thay "cách lấy nơi đến nhỏ nhất", phần stack giữ nguyên như mục 8.
2. `heapq.heappush(adj[a], b)` cho từng vé; `heapq.heappop(adj[u])` để lấy.
3. `while stack: u = stack[-1]; if adj[u]: stack.append(heapq.heappop(adj[u])) else: route.append(stack.pop())`.

**Bài 5**
1. Điểm đầu của đường Euler là đỉnh có `bậc ra - bậc vào = 1` (mục 5.2); nếu là chu trình thì đỉnh nào có cạnh cũng được.
2. Dùng `Counter` (hoặc `defaultdict(int)`) cho hiệu bậc: `diff[a] += 1; diff[b] -= 1`. Hierholzer stack không cần sort.
3. Sau khi có `route` (đã đảo ngược), trả `[[route[i], route[i + 1]] for i in range(len(route) - 1)]`.

**Bài 6**
1. Hierholzer luôn xé hết các vé **đi được tới từ `JFK`** và trả một dãy sân bay; dãy đó chỉ là hành trình thật khi điều kiện bậc thỏa.
2. Với `[["JFK","AAA"],["JFK","BBB"]]` nó trả `["JFK", "BBB", "AAA"]` — đủ độ dài nhưng không có vé `BBB -> AAA`.
3. `r = itinerary_hierholzer(tickets); return r if is_valid_itinerary(tickets, r) else []`. Nếu tồn tại hành trình thì Hierholzer chắc chắn tìm ra cái nhỏ nhất (đúng như bài gốc); nếu không tồn tại thì kết quả của nó **không thể** hợp lệ — nên một lần kiểm tra là đủ.

## 17. Đáp án tham khảo

> Chỉ đọc phần này **sau khi** đã tự làm. So sánh cách nghĩ của bạn với đáp án quan trọng hơn so từng dòng code.

### Đáp án Bài 1

```text
['ATL', 'JFA', 'JFK', 'SFO']
True
['SFO', 'MUC'] ATL ['SFO', 'MUC']
1 False 0 2
```

- Dòng 1: `"JFA"` và `"JFK"` giống nhau ở `"JF"`, ký tự thứ 3 `'A' < 'K'` nên `"JFA"` đứng trước.
- Dòng 2: phần tử đầu bằng nhau, phần tử thứ 2 `"ATL" < "BOS"` là **đủ để kết luận** — `"ZZZ"` hay `"AAA"` phía sau không còn ý nghĩa gì. Đây chính là "chỗ rẽ đầu tiên quyết định" của mục 2.5.
- Dòng 3 là cái bẫy thứ nhất: `adj["JFK"]` ban đầu là `['SFO', 'MUC', 'ATL']`, `pop()` lấy `ATL` (nhỏ nhất, nằm cuối). Nhưng đối số thứ nhất và thứ ba là **cùng một list**, và `print` chỉ in sau khi đã tính xong cả ba đối số — lúc đó list đã bị `pop` rồi, nên cả hai đều in `['SFO', 'MUC']`.
- Dòng 4 là cái bẫy `defaultdict`: lúc đầu `adj` chỉ có key `JFK` (`len = 1`), `"ATL" in adj` là `False` (`in` không tạo key). Nhưng `adj["ATL"]` **tạo** key `ATL` với list rỗng (`len = 0`), nên `len(adj)` cuối dòng là `2`.

### Đáp án Bài 2

`adj` giảm dần: `JFK: [CCC, BBB, AAA]`, `AAA: [JFK]`, `CCC: [JFK]`.

| Bước | Sự kiện | `route` (post-order) |
|---:|---|---|
| 1 | vào `JFK`, xé vé `JFK->AAA` | `[]` |
| 2 | vào `AAA`, xé vé `AAA->JFK` | `[]` |
| 3 | vào `JFK`, xé vé `JFK->BBB` | `[]` |
| 4 | vào `BBB`: hết vé → ghi `BBB` | `[BBB]` |
| 5 | về `JFK`: còn `[CCC]` → xé vé `JFK->CCC` | `[BBB]` |
| 6 | vào `CCC`, xé vé `CCC->JFK` | `[BBB]` |
| 7 | vào `JFK`: hết vé → ghi `JFK` | `[BBB, JFK]` |
| 8 | về `CCC`: hết vé → ghi `CCC` | `[BBB, JFK, CCC]` |
| 9 | về `JFK`: hết vé → ghi `JFK` | `[BBB, JFK, CCC, JFK]` |
| 10 | về `AAA`: hết vé → ghi `AAA` | `[BBB, JFK, CCC, JFK, AAA]` |
| 11 | về `JFK` (ngoài cùng): hết vé → ghi `JFK` | `[BBB, JFK, CCC, JFK, AAA, JFK]` |

Đảo ngược: `JFK AAA JFK CCC JFK BBB`. Ngõ cụt là `BBB` (không có vé bay đi), được ghi vào `route` **đầu tiên** (bước 4) nên đứng **cuối** đáp án; vòng `JFK->CCC->JFK` được "chèn" vào trước nó. Output của cell dưới in đúng các bước này (thụt lề theo độ sâu đệ quy).

Cell dưới chạy thật Bài 1 để đối chiếu, rồi chạy `trace_hierholzer` cho Bài 2.

**Đáp án Bài 3, 4, 5, 6** nằm ở hai cell code cuối. Mỗi hàm đáp án có tên kết thúc bằng `_answer` để không đè lên hàm TODO của bạn.

In [34]:
print(sorted(["SFO", "JFK", "ATL", "JFA"]))
print(["JFK", "ATL", "ZZZ"] < ["JFK", "BOS", "AAA"])
adj = defaultdict(list)
for a, b in sorted([["JFK", "SFO"], ["JFK", "ATL"], ["JFK", "MUC"]], reverse=True):
    adj[a].append(b)
print(adj["JFK"], adj["JFK"].pop(), adj["JFK"])
print(len(adj), "ATL" in adj, len(adj["ATL"]), len(adj))

print("=== Bài 2 ===")
bai2 = [["JFK", "AAA"], ["AAA", "JFK"], ["JFK", "BBB"], ["JFK", "CCC"], ["CCC", "JFK"]]
assert trace_hierholzer(bai2) == itinerary_bruteforce(bai2)

['ATL', 'JFA', 'JFK', 'SFO']
True
['SFO', 'MUC'] ATL ['SFO', 'MUC']
1 False 0 2
=== Bài 2 ===
vào JFK, vé còn: ['CCC', 'BBB', 'AAA']
  xé vé JFK->AAA
  vào AAA, vé còn: ['JFK']
    xé vé AAA->JFK
    vào JFK, vé còn: ['CCC', 'BBB']
      xé vé JFK->BBB
      vào BBB, vé còn: []
      BBB hết vé -> ghi BBB   route = ['BBB']
      xé vé JFK->CCC
      vào CCC, vé còn: ['JFK']
        xé vé CCC->JFK
        vào JFK, vé còn: []
        JFK hết vé -> ghi JFK   route = ['BBB', 'JFK']
      CCC hết vé -> ghi CCC   route = ['BBB', 'JFK', 'CCC']
    JFK hết vé -> ghi JFK   route = ['BBB', 'JFK', 'CCC', 'JFK']
  AAA hết vé -> ghi AAA   route = ['BBB', 'JFK', 'CCC', 'JFK', 'AAA']
JFK hết vé -> ghi JFK   route = ['BBB', 'JFK', 'CCC', 'JFK', 'AAA', 'JFK']
Đảo ngược: ['JFK', 'AAA', 'JFK', 'CCC', 'JFK', 'BBB']


In [35]:
# ---------- Bài 3: Hierholzer đệ quy ----------
def exercise_find_itinerary_answer(tickets):
    adj = defaultdict(list)
    for a, b in sorted(tickets, reverse=True):      # TODO 1: giảm dần -> pop() ra nhỏ nhất
        adj[a].append(b)
    route = []

    def dfs(u):
        while adj[u]:                               # TODO 2: còn vé thì xé vé nhỏ nhất, đi tiếp
            dfs(adj[u].pop())
        route.append(u)                             # TODO 3: hết vé mới ghi (post-order)

    dfs("JFK")                                      # TODO 4: xuất phát JFK, trả bản đảo ngược
    return route[::-1]


# ---------- Bài 4: stack + min-heap ----------
def exercise_itinerary_heap_stack_answer(tickets):
    adj = defaultdict(list)
    for a, b in tickets:                            # không cần sort trước: heap tự lo
        heapq.heappush(adj[a], b)
    stack, route = ["JFK"], []
    while stack:
        u = stack[-1]
        if adj[u]:
            stack.append(heapq.heappop(adj[u]))     # lấy nơi đến nhỏ nhất, O(log k)
        else:
            route.append(stack.pop())
    return route[::-1]


for t, ans in [(ex1, ans1), (ex2, ans2), (ex3, ans3), (ex4, ans4), (bai2, itinerary_bruteforce(bai2))]:
    assert exercise_find_itinerary_answer(t) == ans
    assert exercise_itinerary_heap_stack_answer(t) == ans
random.seed(4)
for _ in range(300):
    t = random_tickets(random.randint(1, 30))
    assert exercise_itinerary_heap_stack_answer(t) == exercise_find_itinerary_answer(t) == itinerary_stack(t)
print("Bài 3, Bài 4: PASS")

Bài 3, Bài 4: PASS


In [36]:
# ---------- Bài 5: LeetCode 2097 ----------
def exercise_valid_arrangement_answer(pairs):
    diff = Counter()
    adj = defaultdict(list)
    for a, b in pairs:
        adj[a].append(b)
        diff[a] += 1                                # bậc ra
        diff[b] -= 1                                # bậc vào
    start = next((x for x in diff if diff[x] == 1), pairs[0][0])   # không có -> chu trình
    stack, route = [start], []
    while stack:                                    # Hierholzer bằng stack, không đệ quy
        u = stack[-1]
        if adj[u]:
            stack.append(adj[u].pop())
        else:
            route.append(stack.pop())
    route.reverse()
    return [[route[i], route[i + 1]] for i in range(len(route) - 1)]


for p in [[[5, 1], [4, 5], [11, 9], [9, 4]], [[1, 3], [3, 2], [2, 1]], [[1, 2], [1, 3], [2, 1]]]:
    got = exercise_valid_arrangement_answer(p)
    assert is_valid_arrangement(p, got)
    print(p, "->", got)
big_pairs = [[i, i + 1] for i in range(100_000)]    # chuỗi thẳng 10^5 cạnh: đệ quy sẽ RecursionError
random.seed(5); random.shuffle(big_pairs)
assert is_valid_arrangement(big_pairs, exercise_valid_arrangement_answer(big_pairs))
print("Bài 5: PASS (kể cả 10^5 cạnh)")


# ---------- Bài 6: không bảo đảm có hành trình ----------
def exercise_itinerary_or_empty_answer(tickets):
    r = itinerary_stack(tickets)                    # bản stack: không lo đệ quy sâu
    return r if is_valid_itinerary(tickets, r) else []


print("Hierholzer thô trên [JFK->AAA, JFK->BBB]:", itinerary_stack([["JFK", "AAA"], ["JFK", "BBB"]]))
assert exercise_itinerary_or_empty_answer([["JFK", "AAA"], ["JFK", "BBB"]]) == []
assert exercise_itinerary_or_empty_answer([["ATL", "SFO"]]) == []
assert exercise_itinerary_or_empty_answer(ex3) == ans3
random.seed(6)
none_count = 0
for _ in range(300):
    t = [random.sample(AIRPORTS, 2) for _ in range(random.randint(1, 6))]
    expected = itinerary_bruteforce(t) or []
    none_count += expected == []
    assert exercise_itinerary_or_empty_answer(t) == expected
print(f"Bài 6: PASS (300 bộ vé ngẫu nhiên, {none_count} bộ không có hành trình)")

[[5, 1], [4, 5], [11, 9], [9, 4]] -> [[11, 9], [9, 4], [4, 5], [5, 1]]
[[1, 3], [3, 2], [2, 1]] -> [[1, 3], [3, 2], [2, 1]]
[[1, 2], [1, 3], [2, 1]] -> [[1, 2], [2, 1], [1, 3]]


Bài 5: PASS (kể cả 10^5 cạnh)
Hierholzer thô trên [JFK->AAA, JFK->BBB]: ['JFK', 'BBB', 'AAA']
Bài 6: PASS (300 bộ vé ngẫu nhiên, 252 bộ không có hành trình)


## 18. Template ghi nhớ và checklist trước khi submit

### Template Hierholzer (đường đi Euler, nhỏ nhất theo từ điển)

```python
adj = defaultdict(list)
for a, b in sorted(edges, reverse=True):   # 1. sort GIẢM dần -> pop() ra nhỏ nhất
    adj[a].append(b)

route = []
def dfs(u):
    while adj[u]:                          # 2. WHILE còn cạnh: xé cạnh nhỏ nhất, đi tiếp
        dfs(adj[u].pop())
    route.append(u)                        # 3. hết cạnh mới ghi (POST-order)

dfs(start)
return route[::-1]                         # 4. ĐẢO ngược
```

Câu thần chú: **"Sort giảm — while xé — hết mới ghi — đảo ngược."** Ngõ cụt hết vé sớm nhất nên được ghi trước, đảo lại thì đứng cuối hành trình.

Bản stack (khi số cạnh lớn): `stack = [start]; while stack: u = stack[-1]; còn cạnh -> stack.append(adj[u].pop()); hết -> route.append(stack.pop())`.

### Checklist trước khi submit

- [ ] Xuất phát từ `"JFK"`, **không** phải từ sân bay nhỏ nhất.
- [ ] Danh sách kề là `list` (giữ vé trùng), **không** phải `set`.
- [ ] Sort **giảm dần** nếu dùng `pop()` (hoặc tăng dần + `popleft()`/heap) — tự hỏi: "phần tử mình lấy ra có phải là nhỏ nhất không?".
- [ ] Dùng `while adj[u]`, không dùng `for v in adj[u]` trong khi đang xóa phần tử của chính list đó.
- [ ] `route.append(u)` đặt **sau** vòng lặp (post-order) và trả `route[::-1]`.
- [ ] Không có `visited` cho sân bay — sân bay được ghé lại nhiều lần; chỉ **vé** là dùng một lần.
- [ ] Kết quả dài đúng `len(tickets) + 1`.
- [ ] Đã thử tay `ex3` (`JFK->KUL`, `JFK->NRT`, `NRT->JFK` → `JFK NRT JFK KUL`): đây là test bắt được hầu hết lỗi mà ví dụ của đề bỏ sót.
- [ ] Độ phức tạp nói được ngay: `O(E log E)` thời gian (do sort), `O(V + E)` bộ nhớ.